# 🌿 Saffron Authenticity: Real vs Fake — end-to-end sample-level cross-validation

**Model**: ConvNeXt-Tiny + Grid Graph-Attention (GNN) + Hybrid Attention Fusion (HAFM), three logit heads,
MC-Dropout uncertainty, 8 colour-preserving test-time views, Wombat-OA weighted beam decoding.

**Novelty**: Wombat-OA-tuned diversity-penalised beam search over test-time views + log-probability fusion.

## 📋 Table of Contents

| § | Section | Skip? |
|---|---------|-------|
| 1 | GPU & environment check | Never |
| 2 | Config | Edit only this cell |
| 3 | Drive mount | Skip if running locally |
| 4 | Write project files | Idempotent, run once |
| 5 | Data preparation | Run once; idempotent |
| 5a | Dataset audit | After §5 |
| 6 | Colour-statistics baseline | Before training |
| 7 | Smoke test | Recommended once (SMOKE=True) |
| 8 | Resume logic | Run on reconnect |
| 9 | Full cross-validation | Main job (~hours on GPU) |
| 10 | Aggregate results | After §9 |
| 11 | Training curves | After §9 |
| 12 | Per-fold results | After §9 |
| 13 | Pooled out-of-fold analysis | After §9 |
| 14 | Tuned decoder parameters | After §9 |
| 15 | Uncertainty check | After §9 |
| 16 | Per-type breakdown | After §9 |
| 17 | Optimiser study | After §9 |
| 18 | Headline dashboard | After §9 |
| 19 | Package results | After §9 |
| 20 | Interpretation guide | Read anytime |
| 21 | Troubleshooting | Read if stuck |
| 22 | Parameters reference | Read anytime |
| 24 | Leave-one-camera-out evaluation | After §9 |
| 25 | Stress-test robustness | After §9 |
| 26 | What to do next | Read anytime |

> **Runtime → Change runtime type → T4 GPU** (or better).
> **Restart-safe**: re-running any cell is safe. Finished folds are skipped automatically (§9).
> **MUST NOT** report results on the original Training/Testing split — it leaks all test samples into training.


## § 1 · GPU & Environment Check

In [ ]:
# § 1 — GPU & Environment Check
# Checks GPU, TF/Keras versions, installs missing packages. Never imports TF in the kernel.
import os, subprocess, sys

ALLOW_CPU = False   # set True in Config cell (§2) — overridden there

# ---- nvidia-smi ----
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if r.returncode != 0:
    msg = "⚠ No GPU found. Set ALLOW_CPU=True in §2 to continue on CPU (very slow)."
    if not ALLOW_CPU:
        raise RuntimeError(msg)
    print(msg)
else:
    print(r.stdout[:600])

# ---- TF / Keras version check via subprocess (never import TF in kernel) ----
_ver = subprocess.run(
    [sys.executable, "-c",
     "import tensorflow as tf, keras; "
     "print(tf.__version__, keras.__version__, "
     "len(tf.config.list_physical_devices('GPU')))"],
    capture_output=True, text=True
)
if _ver.returncode != 0:
    print("TensorFlow not yet installed — will install in §2 / §4.")
    _tf_ver, _keras_ver, _ngpu = None, None, 0
else:
    _parts = _ver.stdout.strip().split()
    _tf_ver, _keras_ver, _ngpu = _parts[0], _parts[1], int(_parts[2])
    print(f"TensorFlow {_tf_ver}  |  Keras {_keras_ver}  |  GPUs visible: {_ngpu}")

# ---- Version requirements ----
from packaging.version import Version
_need_install = False
if _tf_ver and (Version(_tf_ver) < Version("2.16") or Version(_keras_ver) < Version("3.3")):
    print("⚠  TF < 2.16 or Keras < 3.3 detected. Will pip-install below. RESTART RUNTIME ONCE after install.")
    _need_install = True

if _need_install:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "tensorflow>=2.16", "keras>=3.3"],
        check=True
    )
    print("✅ Installed. Please RESTART THE RUNTIME, then re-run from §1.")
    raise SystemExit("Restart the runtime now.")

# ---- GPU memory growth (if GPU present) ----
subprocess.run(
    [sys.executable, "-c",
     "import tensorflow as tf; "
     "[tf.config.experimental.set_memory_growth(g, True) "
     " for g in tf.config.list_physical_devices('GPU')]"],
    capture_output=True
)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
print("§1 complete ✅")


## § 2 · Config (the only cell you edit)

In [ ]:
# § 2 — Config: edit this cell only.
# All other cells read these variables.

# ── Paths ──────────────────────────────────────────────────────────────────────────────
DRIVE_DATA_DIR  = "/content/drive/MyDrive/saffron_data"   # Drive folder containing the two .zip files (or extracted folders)
DRIVE_RUNS_DIR  = "/content/drive/MyDrive/saffron_runs"   # Drive folder for checkpoints (created automatically)

# ── Dataset / training ─────────────────────────────────────────────────────────────────
N_FOLDS       = 5
IMG_SIZE      = 256   # 384 needs BATCH=8 and more GPU RAM
BATCH         = 16    # use 8 if GPU < 16 GB
EPOCHS_WARM   = 3
EPOCHS_FT     = 25
MC_SAMPLES    = 16
POP           = 20    # Wombat-OA population
ITERS         = 25    # Wombat-OA iterations
SEED          = 42

# ── Run control ────────────────────────────────────────────────────────────────────────
SMOKE             = False   # True  → fold 0 only, tiny settings (pipeline test; numbers meaningless)
ALLOW_CPU         = False   # True  → skip GPU check (training will be very slow)
SAVE_WEIGHTS_TO_DRIVE = False  # True → include *.weights.h5 in Drive rsync (~300 MB per fold)

# ── Smoke-mode overrides (applied automatically when SMOKE=True) ────────────────────────
if SMOKE:
    N_FOLDS    = 5          # keep manifest folds, but only run fold 0
    IMG_SIZE   = 128
    BATCH      = 8
    EPOCHS_WARM = 1
    EPOCHS_FT  = 1
    MC_SAMPLES = 4
    POP        = 8
    ITERS      = 4
    print("⚠  SMOKE mode: fold 0 only, tiny settings. Numbers produced are NOT results.")

# ── Derived paths ──────────────────────────────────────────────────────────────────────
import os
COLAB_ROOT  = "/content/saffron-authenticity"
LOCAL_DATA  = os.path.join(COLAB_ROOT, "data")
LOCAL_RUNS  = os.path.join(COLAB_ROOT, "runs", "cv1")
FIGURES_DIR = os.path.join(LOCAL_RUNS, "figures")
os.makedirs(LOCAL_DATA,  exist_ok=True)
os.makedirs(LOCAL_RUNS,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Config: IMG_SIZE={IMG_SIZE}  BATCH={BATCH}  EPOCHS={EPOCHS_WARM}+{EPOCHS_FT}  "
      f"MC={MC_SAMPLES}  POP={POP}  ITERS={ITERS}  SMOKE={SMOKE}")
print(f"COLAB_ROOT = {COLAB_ROOT}")
print(f"DRIVE_DATA = {DRIVE_DATA_DIR}")
print(f"DRIVE_RUNS = {DRIVE_RUNS_DIR}")
print("§2 complete ✅")


## § 3 · Google Drive Mount

In [ ]:
# § 3 — Drive mount.  Skip gracefully when running outside Colab (e.g. local test).
_IN_COLAB = False
try:
    from google.colab import drive as _drive
    _drive.mount("/content/drive", force_remount=False)
    _IN_COLAB = True
    print("Drive mounted ✅")
except ModuleNotFoundError:
    print("⚠ Not running in Colab — Drive mount skipped. Using local paths only.")
    # Override Drive paths to local equivalents for CPU-only local testing
    import os
    DRIVE_DATA_DIR = os.path.join(COLAB_ROOT, "local_data_input")
    DRIVE_RUNS_DIR = os.path.join(COLAB_ROOT, "local_runs_backup")
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)
    print(f"Using local data dir: {DRIVE_DATA_DIR}")
print("§3 complete ✅")


## § 4 · Write Project Files

> **Generated — do not edit these cells manually.**
> They recreate the entire project under `/content/saffron-authenticity/` with the exact same layout as the repository.
> Re-running is idempotent (%%writefile overwrites but does not change logic).


In [ ]:
import os
os.makedirs("/content/saffron-authenticity/scripts", exist_ok=True)
os.makedirs("/content/saffron-authenticity/src", exist_ok=True)
os.makedirs("/content/saffron-authenticity/tests", exist_ok=True)
print('directories ready')

In [ ]:
%%writefile /content/saffron-authenticity/src/saffron_cnn_gnn.py
"""
saffron_cnn_gnn.py  --  TensorFlow/Keras 3 side of the saffron real-vs-fake pipeline.

Differences from the glaucoma notebook (all deliberate):
  * fundus-specific code removed (black-background mask, ungradability head with RANDOM labels)
  * COLOUR-PRESERVING augmentation only (dye colour is the main cue for fake saffron)
  * GNN attention fixed: softmax over each node's neighbours PER SAMPLE (old code normalised over the whole batch)
  * HAFM uses scaled dot-product cross-attention (old tanh-bounded logits could never become sharp)
  * three heads (cnn / gnn / hafm) trained jointly -> three "experts" for log-probability fusion
  * MC-Dropout is NOT baked into the layer; it is switched on only when sampling (training=True on the heads),
    so validation / checkpointing are deterministic and the expensive trunk runs once per view
  * data via tf.data + Keras preprocessing layers (ImageDataGenerator is deprecated and CPU-bound)

Folder layout expected:   root/{train,val,test}/{real,fake}/*.jpg      (fake = 1 = positive class)
"""
from __future__ import annotations
import os
from dataclasses import dataclass, field
import numpy as np
import tensorflow as tf
import keras
from keras import layers

CLASS_NAMES = ["real", "fake"]          # index 0 = real, index 1 = fake (positive)
HEADS = ["cnn", "gnn", "hafm"]
VIEWS = ["id", "hflip", "vflip", "rot90", "rot180", "rot270", "crop90", "crop80"]   # all colour-preserving


@dataclass
class Config:
    img_size: int = 256
    batch: int = 16
    epochs_warm: int = 3            # backbone frozen
    epochs_ft: int = 25             # everything unfrozen
    lr_warm: float = 1e-3
    lr_ft: float = 1e-4
    dropout: float = 0.3
    l2: float = 1e-4                # old notebook used L1=0.1 (loss started at 147) - far too strong
    gnn_dim: int = 128
    gnn_heads: int = 4
    gnn_layers: int = 2
    fusion_dim: int = 128
    mc_samples: int = 16
    imagenet: bool = True
    seed: int = 42
    loss_weights: dict = field(default_factory=lambda: {"cnn": 0.5, "gnn": 0.5, "hafm": 1.0})


# =====================================================================================
# Data
# =====================================================================================
def _count(root, split, cls):
    d = os.path.join(root, split, cls)
    return len([f for f in os.listdir(d) if not f.startswith(".")])


def make_datasets(root, cfg: Config):
    S = cfg.img_size
    common = dict(class_names=CLASS_NAMES, label_mode="binary", image_size=(S, S), batch_size=cfg.batch)
    train_raw = keras.utils.image_dataset_from_directory(os.path.join(root, "train"), shuffle=True, seed=cfg.seed, **common)
    val_raw = keras.utils.image_dataset_from_directory(os.path.join(root, "val"), shuffle=False, **common)
    test_raw = keras.utils.image_dataset_from_directory(os.path.join(root, "test"), shuffle=False, **common)

    n_real, n_fake = _count(root, "train", "real"), _count(root, "train", "fake")
    n = n_real + n_fake
    w_real, w_fake = n / (2 * n_real), n / (2 * n_fake)             # balanced class weights
    print(f"train: real={n_real} fake={n_fake}  weights real={w_real:.2f} fake={w_fake:.2f}")

    # Colour-preserving augmentation. NO channel shift / hue / saturation / strong contrast changes.
    aug = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.5, fill_mode="reflect"),            # +-180 deg: saffron has no canonical orientation
        layers.RandomZoom((-0.15, 0.15), fill_mode="reflect"),
        layers.RandomTranslation(0.05, 0.05, fill_mode="reflect"),
        layers.RandomBrightness(0.05, value_range=(0, 255)),        # mild lighting variation only
    ], name="augment")

    def _train_map(x, y):
        x = aug(x, training=True)
        x = x + tf.random.normal(tf.shape(x), 0.0, 4.0)             # sensor noise, std 4 on the 0-255 scale
        x = tf.clip_by_value(x, 0.0, 255.0)
        sw = tf.where(y[:, 0] > 0.5, w_fake, w_real)
        return x, {h: y for h in HEADS}, sw

    def _eval_map(x, y):
        return x, {h: y for h in HEADS}

    AUTO = tf.data.AUTOTUNE
    train = train_raw.map(_train_map, num_parallel_calls=AUTO).prefetch(AUTO)
    val = val_raw.map(_eval_map, num_parallel_calls=AUTO).prefetch(AUTO)
    return train, val, val_raw, test_raw          # *_raw yield (x, y) and are deterministic -> used for caching


# ---- manifest-based pipeline (sample-level splits, variable-size photos) -------------------------------------------------
def _load_image(path, S, train):
    """Decode a photo of ANY size/aspect -> (S,S,3) float32 in 0-255.
    Square crop (random position when training, centre otherwise) instead of squashing, so thread shapes are not
    distorted.  Training adds random down/up-resampling ('resolution jitter') so the net cannot use image sharpness /
    original resolution as a shortcut (real photos in this dataset are mostly larger than fake ones)."""
    x = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    x.set_shape([None, None, 3])
    h, w = tf.shape(x)[0], tf.shape(x)[1]
    sq = tf.minimum(h, w)
    if train:
        oy = tf.random.uniform([], 0, h - sq + 1, tf.int32); ox = tf.random.uniform([], 0, w - sq + 1, tf.int32)
    else:
        oy, ox = (h - sq) // 2, (w - sq) // 2
    x = tf.image.crop_to_bounding_box(x, oy, ox, sq, sq)
    x = tf.image.resize(tf.cast(x, tf.float32), (S, S), method="bilinear", antialias=True)
    if train:
        m = tf.cast(tf.cast(S, tf.float32) * tf.random.uniform([], 0.4, 1.0), tf.int32)
        x = tf.cond(tf.random.uniform([]) < 0.5,
                    lambda: tf.image.resize(tf.image.resize(x, (m, m), antialias=True), (S, S)), lambda: x)
    x = tf.clip_by_value(x, 0.0, 255.0)
    x.set_shape([S, S, 3])
    return x


def _build_aug():
    # Colour-preserving augmentation. NO channel shift / hue / saturation / strong contrast changes.
    return keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.5, fill_mode="reflect"),            # +-180 deg: saffron has no canonical orientation
        layers.RandomZoom((-0.15, 0.15), fill_mode="reflect"),
        layers.RandomTranslation(0.05, 0.05, fill_mode="reflect"),
        layers.RandomBrightness(0.05, value_range=(0, 255)),        # mild lighting variation only
    ], name="augment")


def make_datasets_from_manifest(train_df, val_df, test_df, cfg: Config):
    """DataFrames need columns: path, label (1 = fake), group.  Returns the same 4 objects as make_datasets()."""
    S, AUTO = cfg.img_size, tf.data.AUTOTUNE
    n_fake = int(train_df.label.sum()); n_real = len(train_df) - n_fake
    w_real, w_fake = len(train_df) / (2 * n_real), len(train_df) / (2 * n_fake)
    print(f"train: real={n_real} fake={n_fake}  weights real={w_real:.2f} fake={w_fake:.2f}")
    aug = _build_aug()

    def raw(df, train):
        ds = tf.data.Dataset.from_tensor_slices((df.path.values.astype(str), df.label.values.astype("float32")))
        if train: ds = ds.shuffle(len(df), seed=cfg.seed, reshuffle_each_iteration=True)
        return ds.map(lambda p, l: (_load_image(p, S, train), l[None]), num_parallel_calls=AUTO).batch(cfg.batch)

    def _train_map(x, y):
        x = aug(x, training=True)
        x = tf.clip_by_value(x + tf.random.normal(tf.shape(x), 0.0, 4.0), 0.0, 255.0)
        return x, {h: y for h in HEADS}, tf.where(y[:, 0] > 0.5, w_fake, w_real)

    train = raw(train_df, True).map(_train_map, num_parallel_calls=AUTO).prefetch(AUTO)
    val_raw, test_raw = raw(val_df, False).cache().prefetch(AUTO), raw(test_df, False).cache().prefetch(AUTO)
    val = val_raw.map(lambda x, y: (x, {h: y for h in HEADS}), num_parallel_calls=AUTO).prefetch(AUTO)
    return train, val, val_raw, test_raw


# =====================================================================================
# Layers
# =====================================================================================
@keras.saving.register_keras_serializable(package="saffron")
class GridGraphAttention(layers.Layer):
    """GAT-style multi-head attention on an HxW patch grid (8-neighbourhood + self loops).

    Message passing:  h_i' = sum_{j in N(i)} alpha_ij W h_j,   alpha_ij = softmax_j( LeakyReLU(a_src.Wh_i + a_dst.Wh_j) )
    The softmax runs over the neighbours j of each node i, independently for every sample and head.
    """

    def __init__(self, units, grid, heads=4, rate=0.1, **kw):
        super().__init__(**kw)
        assert units % heads == 0
        self.units, self.grid, self.heads, self.rate = units, tuple(grid), heads, rate
        self.d = units // heads
        self.W = layers.Dense(units, use_bias=False)
        self.skip = layers.Dense(units, use_bias=False)
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        self.drop = layers.Dropout(rate)

    def build(self, input_shape):
        self.W.build(input_shape)
        self.skip.build(input_shape)
        self.norm.build((None, None, self.units))
        self.a_src = self.add_weight(name="a_src", shape=(self.heads, self.d), initializer="glorot_uniform")
        self.a_dst = self.add_weight(name="a_dst", shape=(self.heads, self.d), initializer="glorot_uniform")
        Hh, Ww = self.grid
        ys, xs = np.divmod(np.arange(Hh * Ww), Ww)
        adj = (np.abs(ys[:, None] - ys[None, :]) <= 1) & (np.abs(xs[:, None] - xs[None, :]) <= 1)
        self.adj_bias = np.where(adj, 0.0, -1e9).astype("float32")      # (N,N) additive mask
        super().build(input_shape)

    def call(self, x, training=None):
        Hh, Ww = self.grid
        N = Hh * Ww
        B = tf.shape(x)[0]
        h = tf.reshape(self.W(x), (B, N, self.heads, self.d))
        e_src = tf.einsum("bnhd,hd->bnh", h, self.a_src)
        e_dst = tf.einsum("bnhd,hd->bnh", h, self.a_dst)
        e = tf.nn.leaky_relu(e_src[:, :, None, :] + e_dst[:, None, :, :], 0.2)     # (B,N,N,heads)
        e = e + tf.constant(self.adj_bias)[None, :, :, None]
        att = self.drop(tf.nn.softmax(e, axis=2), training=training)               # over neighbours j
        out = tf.reshape(tf.einsum("bijh,bjhd->bihd", att, h), (B, N, self.units))
        return tf.nn.gelu(self.norm(out + self.skip(x)))

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], self.units)

    def get_config(self):
        c = super().get_config(); c.update(units=self.units, grid=self.grid, heads=self.heads, rate=self.rate); return c


@keras.saving.register_keras_serializable(package="saffron")
class HybridAttentionFusion(layers.Layer):
    """HAFM: CNN global vector (query) attends over GNN node embeddings (keys/values), then a channel gate and a
    learnable scalar alpha blend the two streams.   Input: [cnn_vec (B,Cc), gnn_nodes (B,N,Cg)]."""

    def __init__(self, dim=128, **kw):
        super().__init__(**kw)
        self.dim = dim
        self.pc, self.pg = layers.Dense(dim), layers.Dense(dim)
        self.gate = layers.Dense(dim, activation="sigmoid")

    def build(self, input_shape):
        cs, gs = input_shape
        self.pc.build(cs); self.pg.build(gs); self.gate.build((None, self.dim))
        self.alpha_raw = self.add_weight(name="alpha_raw", shape=(), initializer="zeros")   # sigmoid(0)=0.5
        super().build(input_shape)

    def call(self, inputs):
        cnn_vec, nodes = inputs
        c, g = self.pc(cnn_vec), self.pg(nodes)                                            # (B,D), (B,N,D)
        attn = tf.nn.softmax(tf.einsum("bnd,bd->bn", g, c) / np.sqrt(self.dim), axis=1)    # (B,N)
        ctx = tf.einsum("bn,bnd->bd", attn, g)
        a = tf.sigmoid(self.alpha_raw)
        return a * c + (1.0 - a) * self.gate(c) * ctx, attn

    def compute_output_shape(self, input_shape):
        return (input_shape[0][0], self.dim), (input_shape[0][0], input_shape[1][1])

    def get_config(self):
        c = super().get_config(); c.update(dim=self.dim); return c


# =====================================================================================
# Model
# =====================================================================================
def _make_head(name, in_dim, cfg: Config):
    h = keras.Sequential([layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.L2(cfg.l2)),
                          layers.Dropout(cfg.dropout),            # MC-dropout source (stochastic only when training=True)
                          layers.Dense(1)], name=f"head_{name}")    # linear -> logit of P(fake)
    h.build((None, in_dim))
    return h


def build_models(cfg: Config):
    S = cfg.img_size
    inp = keras.Input((S, S, 3), name="image")                      # raw 0-255 pixels (ConvNeXt normalises internally)
    base = keras.applications.ConvNeXtTiny(include_top=False, include_preprocessing=True,
                                           weights="imagenet" if cfg.imagenet else None, input_shape=(S, S, 3))
    fmap = base(inp)
    h, w, c = fmap.shape[1:]
    f_cnn = layers.GlobalAveragePooling2D(name="gap")(fmap)
    g = layers.Reshape((h * w, c), name="to_nodes")(fmap)
    for i in range(cfg.gnn_layers):
        g = GridGraphAttention(cfg.gnn_dim, (h, w), heads=cfg.gnn_heads, name=f"gat_{i}")(g)
    f_gnn = layers.GlobalAveragePooling1D(name="gnn_pool")(g)
    f_hafm, attn = HybridAttentionFusion(cfg.fusion_dim, name="hafm")([f_cnn, g])

    heads = {"cnn": _make_head("cnn", c, cfg), "gnn": _make_head("gnn", cfg.gnn_dim, cfg),
             "hafm": _make_head("hafm", cfg.fusion_dim, cfg)}
    feats = {"cnn": f_cnn, "gnn": f_gnn, "hafm": f_hafm}
    model = keras.Model(inp, {k: heads[k](feats[k]) for k in HEADS}, name="saffron_cnn_gnn_hafm")
    trunk = keras.Model(inp, feats, name="trunk")                    # shares ALL layers with `model`
    attn_model = keras.Model(inp, attn, name="hafm_attention")       # for explainability maps (h x w grid)
    return dict(model=model, trunk=trunk, heads=heads, base=base, attn_model=attn_model, grid=(h, w))


def _compile(model, lr, cfg):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss={k: keras.losses.BinaryCrossentropy(from_logits=True) for k in HEADS},
        loss_weights=cfg.loss_weights,
        metrics={k: [keras.metrics.BinaryAccuracy(threshold=0.0, name="acc"),
                     keras.metrics.AUC(from_logits=True, name="auc")] for k in HEADS})


def train(M, train_ds, val_ds, cfg: Config, ckpt="saffron_best.weights.h5"):
    model, base = M["model"], M["base"]
    base.trainable = False                                           # stage 1: heads / GNN / HAFM only
    _compile(model, cfg.lr_warm, cfg)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=cfg.epochs_warm)
    base.trainable = True                                            # stage 2: fine-tune everything
    _compile(model, cfg.lr_ft, cfg)
    cbs = [keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, save_weights_only=True),
           keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
           keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)]
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=cfg.epochs_ft, callbacks=cbs)
    return h1, h2


# =====================================================================================
# Test-time views + MC-dropout cache  (feeds wombat_beam.Cache)
# =====================================================================================
def apply_view(x, name, S):
    if name == "id":     return x
    if name == "hflip":  return tf.image.flip_left_right(x)
    if name == "vflip":  return tf.image.flip_up_down(x)
    if name == "rot90":  return tf.image.rot90(x, 1)
    if name == "rot180": return tf.image.rot90(x, 2)
    if name == "rot270": return tf.image.rot90(x, 3)
    if name.startswith("crop"):
        return tf.image.resize(tf.image.central_crop(x, int(name[4:]) / 100.0), (S, S))
    raise ValueError(name)


def extract_cache(M, raw_ds, cfg: Config, views=VIEWS, T=None, groups=None):
    """One pass over a *deterministic* (shuffle=False) dataset.
    For each image x view x head we store the MC-mean logit and MC std of the logit.
    Only the small heads are re-sampled T times (trunk runs once per view, deterministic)."""
    from wombat_beam import Cache
    T = T or cfg.mc_samples
    trunk, heads, S = M["trunk"], M["heads"], cfg.img_size
    Zs, Us, Ss, Ys = [], [], [], []
    for x, y in raw_ds:
        Zv, Uv, Ev = [], [], []
        for v in views:
            f = trunk(apply_view(x, v, S), training=False)
            per_head = []
            for hname in HEADS:
                s = tf.stack([heads[hname](f[hname], training=True)[:, 0] for _ in range(T)], 0)   # (T,B)
                per_head.append(s)
            zs = tf.stack(per_head, -1)                                                          # (T,B,H)
            Zv.append(tf.reduce_mean(zs, 0)); Uv.append(tf.math.reduce_std(zs, 0))
            Ev.append(tf.math.l2_normalize(f["cnn"], axis=-1))
        E = tf.stack(Ev, 1)                                                                      # (B,V,D)
        Ss.append(tf.clip_by_value(tf.einsum("bvd,bwd->bvw", E, E), 0.0, 1.0).numpy())
        Zs.append(tf.stack(Zv, 1).numpy()); Us.append(tf.stack(Uv, 1).numpy())
        Ys.append(y.numpy()[:, 0].astype(int))
    return Cache(np.concatenate(Zs), np.concatenate(Us), np.concatenate(Ss), np.concatenate(Ys),
                 list(views), list(HEADS), None if groups is None else np.asarray(groups).astype(str))


def run_fold(manifest, fold, n_folds, cfg: Config, out):
    """Outer test = fold k, validation = fold (k+1) % K, train = the rest.  Splits are by physical-sample group.
    Trains, then writes cache_val.npz / cache_test.npz (with sample ids) into `out`."""
    import json, dataclasses
    import pandas as pd
    os.makedirs(out, exist_ok=True)
    kv = (fold + 1) % n_folds
    test_df = manifest[manifest.fold == fold].reset_index(drop=True)
    val_df = manifest[manifest.fold == kv].reset_index(drop=True)
    train_df = manifest[~manifest.fold.isin([fold, kv])].reset_index(drop=True)
    assert not (set(train_df.group) & set(val_df.group)) and not (set(train_df.group) & set(test_df.group)) \
        and not (set(val_df.group) & set(test_df.group)), "sample leakage between splits"
    print(f"fold {fold}: train {len(train_df)} img / {train_df.group.nunique()} samples | "
          f"val {len(val_df)} / {val_df.group.nunique()} | test {len(test_df)} / {test_df.group.nunique()}")
    json.dump(dict(dataclasses.asdict(cfg), fold=fold, n_folds=n_folds), open(f"{out}/config.json", "w"), indent=1)
    keras.utils.set_random_seed(cfg.seed + fold)
    train_ds, val_ds, val_raw, test_raw = make_datasets_from_manifest(train_df, val_df, test_df, cfg)
    M = build_models(cfg)
    h1, h2 = train(M, train_ds, val_ds, cfg, ckpt=f"{out}/best.weights.h5")
    M["model"].save_weights(f"{out}/final.weights.h5")
    pd.DataFrame(h2.history).to_csv(f"{out}/history_finetune.csv", index=False)
    extract_cache(M, val_raw, cfg, groups=val_df.group.values).save(f"{out}/cache_val.npz")
    extract_cache(M, test_raw, cfg, groups=test_df.group.values).save(f"{out}/cache_test.npz")
    keras.backend.clear_session()


# =====================================================================================
# Smoke test:  python saffron_cnn_gnn.py --smoke
# =====================================================================================
def _make_fake_dataset(root, n=12, size=96, seed=0):
    from PIL import Image
    rng = np.random.default_rng(seed)
    for split in ("train", "val", "test"):
        for ci, cls in enumerate(CLASS_NAMES):
            d = os.path.join(root, split, cls); os.makedirs(d, exist_ok=True)
            for i in range(n):
                base = np.array([200, 40, 30]) if ci == 0 else np.array([220, 90, 20])
                img = np.clip(base + rng.normal(0, 25, (size, size, 3)), 0, 255).astype("uint8")
                Image.fromarray(img).save(os.path.join(d, f"{i}.jpg"))


if __name__ == "__main__":
    import sys, tempfile
    if "--smoke" in sys.argv:
        import wombat_beam as wb
        tmp = tempfile.mkdtemp(); _make_fake_dataset(tmp)
        cfg = Config(img_size=128, batch=8, epochs_warm=1, epochs_ft=1, imagenet=False, mc_samples=4)
        train_ds, val_ds, val_raw, test_raw = make_datasets(tmp, cfg)
        M = build_models(cfg); print("grid:", M["grid"]); M["model"].summary(line_length=100)
        train(M, train_ds, val_ds, cfg, ckpt=os.path.join(tmp, "best.weights.h5"))
        cv, ct = extract_cache(M, val_raw, cfg), extract_cache(M, test_raw, cfg)
        print("cache", cv.Z.shape, cv.U.shape, cv.Sim.shape, cv.y[:6])
        table, info = wb.evaluate_all(cv, ct, pop=6, iters=3)
        print(table[["acc", "f1_fake", "auc"]].round(3))
        print("SMOKE TEST OK")


In [ ]:
print('wrote /content/saffron-authenticity/src/saffron_cnn_gnn.py')

In [ ]:
%%writefile /content/saffron-authenticity/src/wombat_beam.py
"""
wombat_beam.py  --  pure NumPy/SciPy/sklearn (no TensorFlow needed)

Contents
--------
1. WombatOptimizer      : Wombat Optimization Algorithm (Benmamoun et al., Mathematics 2024, 12(7):1059),
                          Eq.(2),(4)-(8): foraging phase (exploration) + tunnel-escape phase (exploitation).
                          Named "Wombat-OA" here to avoid confusion with the Whale OA (also abbreviated WOA).
2. RandomSearch         : equal-budget baseline optimizer (reviewers WILL ask for this).
3. Cache                : cached per-(image, view, head) logits / MC-uncertainty / view-similarity.
4. decode()             : weighted, diversity-aware beam search over test-time views
                          + log-probability (log-opinion-pool) fusion of heads and views.
5. metrics + baselines  : everything needed for the ablation table.

Notation
--------
N images, V test-time views, H heads (cnn, gnn, hafm).  Positive class = 1 = FAKE (adulterated).
"""
from __future__ import annotations
import numpy as np
from dataclasses import dataclass
from scipy.special import expit, logsumexp
from sklearn.metrics import roc_auc_score


# =====================================================================================
# 1. Wombat Optimization Algorithm
# =====================================================================================
class WombatOptimizer:
    """Minimises `fitness(x)` over the box [lb, ub].

    Phase 1 (foraging / exploration): each wombat moves toward a randomly chosen *better* wombat
        x' = x + r * (x_forage - I * x),   I in {1,2},  r ~ U(0,1)           (paper Eq.5)
    Phase 2 (escape to tunnel / exploitation): shrinking local random step
        x' = x + (1 - 2r) * (ub - lb) / t,  t = iteration counter            (paper Eq.7)
    Each phase uses greedy replacement (Eq.6, Eq.8).
    """

    def __init__(self, fitness, lb, ub, pop_size=20, iters=30, seed=0, verbose=False):
        self.f = fitness
        self.lb = np.asarray(lb, float)
        self.ub = np.asarray(ub, float)
        self.N, self.T = int(pop_size), int(iters)
        self.rng = np.random.default_rng(seed)
        self.verbose = verbose
        self.n_evals = 0

    def _eval(self, x):
        self.n_evals += 1
        return float(self.f(x))

    def run(self):
        rng, lb, ub, m = self.rng, self.lb, self.ub, self.lb.size
        X = lb + rng.random((self.N, m)) * (ub - lb)              # Eq.(2)
        F = np.array([self._eval(x) for x in X])
        hist = [F.min()]
        for t in range(1, self.T + 1):
            for i in range(self.N):
                # ---- Phase 1: foraging (exploration) ----
                better = np.where(F < F[i])[0]                    # Eq.(4): candidate forage positions
                if better.size:
                    k = rng.choice(better)
                    I = rng.integers(1, 3, size=m)                # randomly 1 or 2
                    r = rng.random(m)
                    xn = np.clip(X[i] + r * (X[k] - I * X[i]), lb, ub)   # Eq.(5)
                    fn = self._eval(xn)
                    if fn <= F[i]:                                # Eq.(6)
                        X[i], F[i] = xn, fn
                # ---- Phase 2: escape to tunnel (exploitation) ----
                r = rng.random(m)
                xn = np.clip(X[i] + (1 - 2 * r) * (ub - lb) / t, lb, ub)  # Eq.(7)
                fn = self._eval(xn)
                if fn <= F[i]:                                    # Eq.(8)
                    X[i], F[i] = xn, fn
            hist.append(F.min())
            if self.verbose:
                print(f"[Wombat-OA] iter {t:3d}/{self.T}  best={F.min():.5f}  evals={self.n_evals}")
        b = int(np.argmin(F))
        return X[b].copy(), float(F[b]), np.array(hist)


class RandomSearch:
    """Same interface, same evaluation budget -> fair optimizer baseline."""

    def __init__(self, fitness, lb, ub, budget=1000, seed=0):
        self.f, self.lb, self.ub = fitness, np.asarray(lb, float), np.asarray(ub, float)
        self.budget, self.rng = int(budget), np.random.default_rng(seed)

    def run(self):
        best_x, best_f, hist = None, np.inf, []
        for _ in range(self.budget):
            x = self.lb + self.rng.random(self.lb.size) * (self.ub - self.lb)
            fx = float(self.f(x))
            if fx < best_f:
                best_x, best_f = x, fx
            hist.append(best_f)
        return best_x, best_f, np.array(hist)


# =====================================================================================
# 2. Cache of model outputs (computed once with TF, reused by every optimizer evaluation)
# =====================================================================================
@dataclass
class Cache:
    Z: np.ndarray          # (N,V,H) MC-mean logits (fake-vs-real), per view & head
    U: np.ndarray          # (N,V,H) MC std of logits  (epistemic uncertainty)
    Sim: np.ndarray        # (N,V,V) cosine similarity between view embeddings, clipped to [0,1]
    y: np.ndarray          # (N,) 1 = fake
    view_names: list
    head_names: list
    groups: np.ndarray | None = None   # (N,) physical-sample id of each image (for sample-level aggregation)

    @property
    def shape(self):
        return self.Z.shape

    def save(self, path):
        extra = {} if self.groups is None else {"groups": np.asarray(self.groups).astype(str)}
        np.savez_compressed(path, Z=self.Z, U=self.U, Sim=self.Sim, y=self.y,
                            view_names=np.array(self.view_names), head_names=np.array(self.head_names), **extra)

    @staticmethod
    def load(path):
        d = np.load(path, allow_pickle=False)
        g = d["groups"] if "groups" in d.files else None
        return Cache(d["Z"], d["U"], d["Sim"], d["y"], list(d["view_names"]), list(d["head_names"]), g)

    @staticmethod
    def synthetic(N=600, V=8, H=3, seed=0, sep=1.2):
        """Toy cache for unit-testing the decoder/optimizer without any model."""
        rng = np.random.default_rng(seed)
        groups = np.arange(N) // 3                                  # 3 "shots" per physical sample
        y = rng.integers(0, 2, groups.max() + 1)[groups]
        s = 2 * y - 1
        view_noise = rng.uniform(0.6, 2.2, V)                    # some views are less reliable
        head_noise = rng.uniform(0.8, 1.6, H)
        Z = np.zeros((N, V, H)); U = np.zeros((N, V, H))
        for v in range(V):
            for h in range(H):
                sd = view_noise[v] * head_noise[h]
                Z[:, v, h] = sep * s + rng.normal(0, sd, N)
                U[:, v, h] = np.abs(rng.normal(0.15 * sd, 0.05, N)) + 0.02
        E = rng.normal(size=(N, V, 16)) + rng.normal(size=(N, 1, 16))   # shared component -> correlated views
        E /= np.linalg.norm(E, axis=-1, keepdims=True)
        Sim = np.clip(np.einsum("nvd,nwd->nvw", E, E), 0, 1)
        return Cache(Z, U, Sim, y, [f"v{i}" for i in range(V)], ["cnn", "gnn", "hafm"][:H], groups.astype(str))


# =====================================================================================
# 3. Weighted beam decoding + log-probability fusion
# =====================================================================================
def param_spec(H, V):
    """Names and box bounds of the decision vector optimised by Wombat-OA."""
    spec = [(f"a_{i}", 0.0, 1.0) for i in range(H)]            # head weights   (log-opinion-pool weights)
    spec += [(f"tau_{i}", 0.5, 3.0) for i in range(H)]         # head temperatures (calibration)
    spec += [("beta", 0.0, 4.0),                               # view reliability: w = exp(-beta * uncertainty)
             ("gamma", 0.0, 2.0),                              # diversity (redundancy) penalty in beam
             ("alpha", 0.0, 1.0),                              # length-normalisation exponent
             ("beam", 1.0, 8.0),                               # beam width B       (rounded)
             ("depth", 1.0, float(V)),                         # decoding depth L   (rounded)
             ("kappa", 0.05, 2.0),                             # sharpness of final fused posterior
             ("theta", 0.2, 0.8)]                              # decision threshold on p(fake)
    names = [s[0] for s in spec]
    return names, np.array([s[1] for s in spec]), np.array([s[2] for s in spec])


def unpack(x, H, V):
    names, _, _ = param_spec(H, V)
    d = dict(zip(names, np.asarray(x, float)))
    a = np.array([d[f"a_{i}"] for i in range(H)]) + 1e-6
    return dict(a=a / a.sum(),
                tau=np.array([d[f"tau_{i}"] for i in range(H)]),
                beta=d["beta"], gamma=d["gamma"], alpha=d["alpha"],
                beam=int(round(d["beam"])), depth=int(np.clip(round(d["depth"]), 1, V)),
                kappa=d["kappa"], theta=d["theta"])


def default_params(H, V):
    """Hand-set, un-tuned configuration (ablation row 'beam, no optimiser')."""
    names, lb, ub = param_spec(H, V)
    d = {f"a_{i}": 1.0 / H for i in range(H)}
    d.update({f"tau_{i}": 1.0 for i in range(H)})
    d.update(beta=1.0, gamma=0.5, alpha=1.0, beam=4, depth=min(V, 5), kappa=1.0, theta=0.5)
    return np.array([d[n] for n in names])


def _softplus(x):
    return np.logaddexp(0.0, x)


def _beam_one_class(ell, w, Sim, B, L, gamma, alpha):
    """Vectorised (over images) beam search for ONE class hypothesis.

    ell : (N,V) log q_v(c)   -- fused per-view log-probability of this class (<= 0)
    w   : (N,V) view weights in (0,1]
    Sim : (N,V,V) non-negative view-similarity
    A hypothesis is an ordered set of views.  Extending hypothesis h by view v adds
          w_v * ell_v  -  gamma * max_{u in h} Sim(v,u)
    (first view has no redundancy penalty).  Beams are ranked by length-normalised score
          S / (sum_w)^alpha ,  duplicates (same view-set reached by different orders) are merged.
    Returns the (N,k) length-normalised scores of the final beam (-inf = dead slot).
    """
    N, V = ell.shape
    gain = w * ell
    score = np.zeros((N, 1)); wsum = np.zeros((N, 1))
    used = np.zeros((N, 1, V), bool)
    mask = np.zeros((N, 1), np.int64)
    maxsim = np.zeros((N, 1, V))
    bits = (1 << np.arange(V)).astype(np.int64)
    rows = np.arange(N)[:, None]
    final = None
    for _ in range(L):
        Bc = score.shape[1]
        cand = score[:, :, None] + gain[:, None, :] - gamma * maxsim          # (N,Bc,V)
        cand_w = wsum[:, :, None] + w[:, None, :]
        cand = np.where(used, -np.inf, cand)
        norm = cand / np.power(np.maximum(cand_w, 1e-9), alpha)
        keys = mask[:, :, None] | bits[None, None, :]
        M = Bc * V
        norm_f, keys_f = norm.reshape(N, M), keys.reshape(N, M)
        # --- merge duplicates: sort by (view-set key, -score), keep first of each key ---
        order = np.lexsort((-norm_f, keys_f), axis=1)
        ks = np.take_along_axis(keys_f, order, 1)
        ns = np.take_along_axis(norm_f, order, 1)
        first = np.ones_like(ks, bool)
        first[:, 1:] = ks[:, 1:] != ks[:, :-1]
        ns = np.where(first, ns, -np.inf)
        k = min(B, M)
        top = np.argsort(-ns, axis=1)[:, :k]
        sel = np.take_along_axis(order, top, 1)                               # flat (parent*V+view)
        sel_ns = np.take_along_axis(ns, top, 1)
        parent, v = sel // V, sel % V
        score = cand.reshape(N, M)[rows, sel]
        wsum = cand_w.reshape(N, M)[rows, sel]
        used = used[rows, parent].copy()
        np.put_along_axis(used, v[:, :, None], True, axis=2)
        maxsim = np.maximum(maxsim[rows, parent], Sim[rows, v])               # (N,k,V)
        mask = keys_f[rows, sel]
        score = np.where(np.isfinite(sel_ns), score, -np.inf)
        final = sel_ns
    return final


def decode(cache: Cache, p: dict) -> np.ndarray:
    """Returns p(fake) for every image.

    1. Head fusion in log-domain (log-opinion pool), per view:
          l_c(v) = sum_h a_h * log sigma_c(z_vh / tau_h),  then renormalise over c.
    2. View reliability weight  w_v = exp(-beta * sum_h a_h * std_vh / tau_h)   (MC-dropout uncertainty).
    3. Class-conditional weighted beam search over view sets (diversity-penalised).
    4. Log-prob fusion over the final beam:  F_c = logmeanexp_h S_norm(h);  p(fake) = sigmoid(kappa (F_1 - F_0)).
    """
    Z, U, Sim = cache.Z, cache.U, cache.Sim
    N, V, H = Z.shape
    a, tau = p["a"], p["tau"]
    zt = Z / tau                                                       # (N,V,H)
    lp1 = -_softplus(-zt)                                              # log sigma(z/tau)
    lp0 = -_softplus(zt)
    l1 = (lp1 * a).sum(-1); l0 = (lp0 * a).sum(-1)
    lse = np.logaddexp(l0, l1)
    l1, l0 = l1 - lse, l0 - lse                                        # proper per-view distribution
    unc = ((U / tau) * a).sum(-1)
    w = np.exp(-p["beta"] * unc)
    F = []
    for ell in (l0, l1):
        S = _beam_one_class(ell, w, Sim, p["beam"], p["depth"], p["gamma"], p["alpha"])
        cnt = np.isfinite(S).sum(1)
        F.append(logsumexp(S, axis=1) - np.log(np.maximum(cnt, 1)))
    return expit(p["kappa"] * (F[1] - F[0]))


# =====================================================================================
# 4. Metrics
# =====================================================================================
def ece_score(p, y, bins=10):
    idx = np.minimum((p * bins).astype(int), bins - 1)
    e = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)


def metrics(y, p, thr=0.5):
    y = np.asarray(y).astype(int); pred = (p >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    sens = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1); prec = tp / max(tp + fp, 1)
    f1 = 2 * prec * sens / max(prec + sens, 1e-12)
    pc = np.clip(p, 1e-7, 1 - 1e-7)
    return dict(acc=(tp + tn) / len(y), bal_acc=(sens + spec) / 2, precision=prec,
                recall_fake=sens, specificity=spec, f1_fake=f1,
                auc=float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else float("nan"),
                ece=ece_score(p, y), nll=float(-(y * np.log(pc) + (1 - y) * np.log(1 - pc)).mean()),
                brier=float(((p - y) ** 2).mean()), fn=fn, fp=fp)


def _macro_f1_fast(y, pred):
    f = []
    for c in (0, 1):
        tp = ((pred == c) & (y == c)).sum(); fp = ((pred == c) & (y != c)).sum(); fn = ((pred != c) & (y == c)).sum()
        f.append(2 * tp / max(2 * tp + fp + fn, 1))
    return float(np.mean(f))


def aggregate_by_group(p, y, groups, eps=1e-6):
    """Sample-level decision: fuse all photos (shots) of one physical sample by averaging their logits
    (= log-probability fusion across shots).  Returns (p_sample, y_sample)."""
    g = np.asarray(groups); u, inv = np.unique(g, return_inverse=True)
    lg = np.log(np.clip(p, eps, 1 - eps) / (1 - np.clip(p, eps, 1 - eps)))
    m = np.bincount(inv, weights=lg) / np.bincount(inv)
    ys = np.zeros(len(u)); ys[inv] = y
    return expit(m), ys.astype(int)


def bootstrap_ci(y, p, thr=0.5, key="acc", n=1000, seed=0):
    rng = np.random.default_rng(seed); y = np.asarray(y); vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        vals.append(metrics(y[i], p[i], thr)[key])
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


# =====================================================================================
# 5. Fitness + tuning
# =====================================================================================
def make_fitness(cache: Cache, lam_ece=0.5, lam_fn=0.25):
    """Minimise  (1 - macroF1) + lam_ece*ECE + lam_fn*FNR   on the VALIDATION cache.
    FNR (fakes accepted as real) is penalised extra because that is the costly error in authentication."""
    N, V, H = cache.shape
    y = cache.y.astype(int)

    def fit(x):
        p = unpack(x, H, V)
        prob = decode(cache, p)
        pred = (prob >= p["theta"]).astype(int)
        fnr = ((pred == 0) & (y == 1)).sum() / max((y == 1).sum(), 1)
        return (1 - _macro_f1_fast(y, pred)) + lam_ece * ece_score(prob, y) + lam_fn * fnr
    return fit


def tune(cache_val: Cache, method="wombat", pop=20, iters=25, seed=0, verbose=False, **fit_kw):
    N, V, H = cache_val.shape
    names, lb, ub = param_spec(H, V)
    fit = make_fitness(cache_val, **fit_kw)
    if method == "wombat":
        opt = WombatOptimizer(fit, lb, ub, pop_size=pop, iters=iters, seed=seed, verbose=verbose)
        x, f, hist = opt.run()
    elif method == "random":
        x, f, hist = RandomSearch(fit, lb, ub, budget=pop + 2 * pop * iters, seed=seed).run()
    else:
        raise ValueError(method)
    return x, f, hist


# =====================================================================================
# 6. Baselines for the ablation table (all on cached outputs -> identical inputs everywhere)
# =====================================================================================
def b_single(c: Cache, head, view=0):
    return expit(c.Z[:, view, c.head_names.index(head)])


def b_late_fusion_identity(c: Cache):                      # prob-average of heads, identity view only
    return expit(c.Z[:, 0, :]).mean(-1)


def b_tta_mean_logit(c: Cache, head):                      # plain TTA
    return expit(c.Z[:, :, c.head_names.index(head)].mean(1))


def b_tta_topk_conf(c: Cache, head, k=4, temp=2.0):        # the old notebook's "TTA-Beam" (top-k by confidence)
    p = expit(c.Z[:, :, c.head_names.index(head)])
    conf = np.maximum(p, 1 - p)
    idx = np.argsort(-conf, axis=1)[:, :k]
    pk, ck = np.take_along_axis(p, idx, 1), np.take_along_axis(conf, idx, 1)
    w = np.exp(ck * temp); w /= w.sum(1, keepdims=True)
    return (w * pk).sum(1)


def b_logprob_fusion_no_beam(c: Cache):                    # equal-weight log-prob fusion over ALL heads and views
    return expit(c.Z.mean((1, 2)))


def evaluate_all(val: Cache, test: Cache, pop=20, iters=25, seed=0, extra_rows=None):
    """Ablation table on TEST. Optimiser/threshold use VAL only."""
    import pandas as pd
    N, V, H = test.shape
    rows = {}
    for h in test.head_names:
        rows[f"single view, {h}"] = (b_single(test, h), 0.5)
    rows["late fusion (prob avg), identity view"] = (b_late_fusion_identity(test), 0.5)
    rows["TTA mean-logit, hafm"] = (b_tta_mean_logit(test, test.head_names[-1]), 0.5)
    rows["TTA top-k confidence (old), hafm"] = (b_tta_topk_conf(test, test.head_names[-1]), 0.5)
    rows["log-prob fusion, equal weights, no beam"] = (b_logprob_fusion_no_beam(test), 0.5)
    pd0 = unpack(default_params(H, V), H, V)
    rows["weighted beam + log-prob fusion (untuned)"] = (decode(test, pd0), pd0["theta"])
    xr, fr, _ = tune(val, "random", pop, iters, seed)
    pr = unpack(xr, H, V); rows["weighted beam + log-prob fusion + random search"] = (decode(test, pr), pr["theta"])
    xw, fw, hist = tune(val, "wombat", pop, iters, seed)
    pw = unpack(xw, H, V); rows["PROPOSED: Wombat-OA + weighted beam + log-prob fusion"] = (decode(test, pw), pw["theta"])
    if extra_rows:                                       # e.g. {"colour-statistics random forest": (p, 0.5)}
        rows.update(extra_rows)
    table = pd.DataFrame({k: metrics(test.y, p, t) for k, (p, t) in rows.items()}).T
    info = dict(params=pw, fitness=fw, history=hist, params_random=pr, probs={k: p for k, (p, _) in rows.items()},
                thr={k: t for k, (_, t) in rows.items()})
    if test.groups is not None:                          # one decision per physical sample (all shots fused)
        agg = {k: aggregate_by_group(p, test.y, test.groups) for k, (p, _) in rows.items()}
        info["sample_table"] = pd.DataFrame({k: metrics(ys, ps, rows[k][1]) for k, (ps, ys) in agg.items()}).T
    return table, info


# =====================================================================================
# 7. Optimiser comparison over many seeds (needed to claim "Wombat-OA is better than X")
# =====================================================================================
def compare_optimizers(val: Cache, seeds=range(10), pop=20, iters=25, **fit_kw):
    """Runs Wombat-OA and equal-budget Random Search with the same seeds on the validation cache.
    Returns best-fitness arrays, convergence curves and a one-sided Wilcoxon signed-rank p-value
    (H1: Wombat-OA reaches lower fitness than random search)."""
    from scipy.stats import wilcoxon
    fw, fr, hw, hr = [], [], [], []
    for s in seeds:
        _, f1, h1 = tune(val, "wombat", pop, iters, s, **fit_kw)
        _, f2, h2 = tune(val, "random", pop, iters, s, **fit_kw)
        fw.append(f1); fr.append(f2); hw.append(h1); hr.append(h2)
    fw, fr = np.array(fw), np.array(fr)
    try:
        p = float(wilcoxon(fw, fr, alternative="less").pvalue)
    except ValueError:                                   # all differences zero
        p = float("nan")
    return dict(wombat=fw, random=fr, hist_wombat=hw, hist_random=hr, wilcoxon_p=p)


# =====================================================================================
# Self-test on synthetic data
# =====================================================================================
if __name__ == "__main__":
    import time
    val, test = Cache.synthetic(seed=1), Cache.synthetic(seed=2)
    t0 = time.time()
    table, info = evaluate_all(val, test, pop=12, iters=10, seed=0)
    import pandas as pd
    pd.set_option("display.width", 200)
    print(table[["acc", "f1_fake", "recall_fake", "specificity", "auc", "ece"]].round(4))
    print("tuned params:", {k: (np.round(v, 3) if hasattr(v, 'shape') else v) for k, v in info["params"].items()})
    print(f"elapsed {time.time() - t0:.1f}s")


In [ ]:
print('wrote /content/saffron-authenticity/src/wombat_beam.py')

In [ ]:
%%writefile /content/saffron-authenticity/src/color_baseline.py
"""Colour-statistics baseline.  On this dataset, mean/std of RGB+HSV alone already separates real from fake very well
(group-CV AUC ~0.94 in our check), so ANY deep-learning claim must be compared against this row."""
import os
import numpy as np
from PIL import Image
from sklearn.ensemble import RandomForestClassifier


def features(paths, cache=None):
    if cache and os.path.exists(cache):
        d = np.load(cache, allow_pickle=True)
        if list(d["paths"]) == list(paths): return d["X"]
    X = []
    for p in paths:
        im = Image.open(p).convert("RGB"); w, h = im.size; s = min(w, h)
        im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((64, 64))
        a = np.asarray(im, float) / 255; v = np.asarray(im.convert("HSV"), float) / 255
        X.append([*a.mean((0, 1)), *a.std((0, 1)), *v.mean((0, 1)), *v.std((0, 1))])
    X = np.array(X)
    if cache: np.savez(cache, paths=np.array(paths), X=X)
    return X


def fit_predict(Xtr, ytr, Xte, seed=0):
    m = RandomForestClassifier(300, min_samples_leaf=2, random_state=seed, class_weight="balanced").fit(Xtr, ytr)
    return m.predict_proba(Xte)[:, 1]


In [ ]:
print('wrote /content/saffron-authenticity/src/color_baseline.py')

In [ ]:
%%writefile /content/saffron-authenticity/scripts/_path.py
import os, sys
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
sys.path.insert(0, os.path.join(ROOT, "src"))


In [ ]:
print('wrote /content/saffron-authenticity/scripts/_path.py')

In [ ]:
%%writefile /content/saffron-authenticity/scripts/prepare_dataset.py
"""Build a leakage-free manifest + sample-level K-fold splits from your Training/Testing zips (or folders).

Your data looks like   Training/{Fake,Real}/<id>[ <TYPE>]_<shot>.jpg   and   Testing/{Fake,Real}/...
Several photos (shots) belong to the SAME physical sample, and the same samples appear in both Training and Testing.
So the original Training/Testing split leaks.  This script
  1. extracts the zips / reads the folders,
  2. parses the filenames   '12_3.jpg' -> sample 12, shot 3 ;  '17 C_2.jpg' -> sample 17, type C, shot 2,
  3. finds exact duplicate files and MERGES the samples that share a photo into one group (dropping the repeat copy),
  4. makes stratified GROUP K-fold splits (all shots of a sample, plus merged duplicates, always in the same fold),
  5. writes data/manifest.csv   (columns: path,label,cls,type,sample,group,shot,orig_split,fold)  and prints a report.

   python scripts/prepare_dataset.py --src data/Test_dataset-....zip data/train_dataset-....zip --folds 5
"""
import _path  # noqa
import argparse, glob, os, re, zipfile, hashlib, collections, sys
import numpy as np, pandas as pd
from PIL import Image, ImageOps
from sklearn.model_selection import StratifiedGroupKFold

ap = argparse.ArgumentParser()
ap.add_argument("--src", nargs="+", required=True, help="zip files and/or folders containing .../{Fake,Real}/*.jpg")
ap.add_argument("--data", default="data"); ap.add_argument("--folds", type=int, default=5); ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--max-side", type=int, default=768, help="working copies: shorter side is capped at this many pixels (never upscaled)")
a = ap.parse_args()
sources = []
for src in a.src:
    if glob.has_magic(src):
        matches = sorted(glob.glob(src))
        if not matches: ap.error(f"--src pattern matched no files: {src}")
        sources.extend(matches)
    else:
        sources.append(src)
a.src = sources
EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
RX = re.compile(r"^(\d+)(?:\s*([A-Za-z]+))?_(\d+)\.\w+$")

# ---- 1. collect files -----------------------------------------------------------------------------------------------
roots = []
for s in a.src:
    if s.lower().endswith(".zip"):
        dst = os.path.join(a.data, "extracted", os.path.splitext(os.path.basename(s))[0])
        if not os.path.isdir(dst):
            print("extracting", s, "->", dst); zipfile.ZipFile(s).extractall(dst)
        roots.append(dst)
    else:
        roots.append(s)
rows = []
for root in roots:
    for dp, _, fs in os.walk(root):
        cls = os.path.basename(dp).lower()
        if cls not in ("fake", "real"): continue
        parent_name = os.path.basename(os.path.dirname(dp)).lower()          # e.g. "Training" / "Testing"
        split = "test" if "test" in parent_name else "train" if "train" in parent_name else "unknown"
        for f in sorted(fs):
            if not f.lower().endswith(EXT) or f.startswith("."): continue
            m = RX.match(f)
            sid, typ, shot = (m[1], (m[2] or "-").upper(), int(m[3])) if m else (os.path.splitext(f)[0], "-", 0)
            rows.append(dict(path=os.path.join(dp, f), label=int(cls == "fake"), cls=cls, type=typ,
                             sample=f"{cls}-{typ}-{sid}", shot=shot, orig_split=split))
df = pd.DataFrame(rows)
if df.empty: sys.exit("no images found: expected folders named Fake / Real")
print(f"found {len(df)} images: {df.groupby('cls').size().to_dict()}  | physical samples: {df['sample'].nunique()}")

# ---- 2. report the leakage of the ORIGINAL split ---------------------------------------------------------------------
tr, te = set(df[df.orig_split == "train"]["sample"]), set(df[df.orig_split == "test"]["sample"])
if tr and te:
    print(f"[LEAK] original split: {len(tr & te)} of {len(te)} test samples ALSO appear in train (different shots of the same sample)")

# ---- 3. exact duplicates -> merge groups, drop repeat copies -----------------------------------------------------------
df["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in df.path]
parent = {s: s for s in df["sample"]}
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
first = {}
for s, h in zip(df["sample"], df.md5):
    if h in first: parent[find(s)] = find(first[h])
    else: first[h] = s
n_dup = int(df.duplicated("md5").sum())
if n_dup: print(f"[DUP] {n_dup} exact duplicate file(s) found -> merged their samples into one group and dropped the repeat copy")
df = df.drop_duplicates("md5").copy()
roots_ = {s: find(s) for s in df["sample"]}; names = {r: f"G{i:03d}" for i, r in enumerate(sorted(set(roots_.values())))}
df["group"] = df["sample"].map(lambda s: names[roots_[s]])
bad = df.groupby("group").label.nunique(); bad = bad[bad > 1]
if len(bad): sys.exit(f"[ERROR] groups mixing real and fake (check duplicates): {list(bad.index)}")
print(f"{df.group.nunique()} groups (physical samples after merging) | per class: {df.groupby('cls').group.nunique().to_dict()}")

# ---- 4. stratified group K-fold -----------------------------------------------------------------------------------------
df = df.reset_index(drop=True); df["fold"] = -1
for k, (_, te_idx) in enumerate(StratifiedGroupKFold(a.folds, shuffle=True, random_state=a.seed).split(df, df.label, df.group)):
    df.loc[te_idx, "fold"] = k
assert (df.fold >= 0).all() and df.groupby("group").fold.nunique().max() == 1
rep = df.groupby("fold").agg(images=("path", "size"), groups=("group", "nunique"), fake_frac=("label", "mean"))
rep["types(fake)"] = [dict(collections.Counter(df[(df.fold == k) & (df.label == 1)].type)) for k in rep.index]
print("\nfolds (each fold is the TEST set once; the next fold is VAL; the rest TRAIN):\n", rep.round(2).to_string())

# ---- 5. working copies: EXIF-rotated, shorter side capped (3024x3024 originals are slow and memory hungry) ----------------
img_dir = os.path.join(a.data, "images"); new_paths = []
for _, r in df.iterrows():
    dst_dir = os.path.join(img_dir, r.cls); os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, f"{r.group}__{os.path.splitext(os.path.basename(r.path))[0]}.jpg")
    if not os.path.exists(dst):
        with Image.open(r.path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB"); w, h = im.size; sc = a.max_side / min(w, h)
            if sc < 1: im = im.resize((round(w * sc), round(h * sc)), Image.LANCZOS)
            im.save(dst, quality=95)
    new_paths.append(dst)
df["orig_path"] = df.path; df["path"] = new_paths
print(f"wrote {len(df)} working copies to {img_dir} (shorter side <= {a.max_side}px)")

df = df.drop(columns=["md5"]); out = os.path.join(a.data, "manifest.csv"); df.to_csv(out, index=False)
print(f"\nwrote {out}  ({len(df)} images)")
if df.groupby("fold").group.nunique().min() < 8:
    print("[NOTE] fewer than 8 groups in a fold: metrics per fold will be very noisy -> report mean +- std over folds")


In [ ]:
print('wrote /content/saffron-authenticity/scripts/prepare_dataset.py')

In [ ]:
%%writefile /content/saffron-authenticity/scripts/run_cv.py
"""Sample-level K-fold cross-validation:  train -> cache -> Wombat-OA beam decoding -> ablation, for every fold,
then aggregate (mean +- std over folds, plus pooled out-of-fold results).

   python scripts/run_cv.py --manifest data/manifest.csv --out runs/cv1                 # all folds
   python scripts/run_cv.py --manifest data/manifest.csv --out runs/cv1 --folds 0       # just fold 0 (quick check)
   python scripts/run_cv.py --manifest data/manifest.csv --out runs/cv1 --aggregate-only
Re-running resumes: folds whose caches already exist are not retrained (use --retrain to force).
"""
import _path  # noqa
import argparse, os, sys, json, subprocess
import numpy as np, pandas as pd
import wombat_beam as wb
import color_baseline as cb

ap = argparse.ArgumentParser()
ap.add_argument("--manifest", default="data/manifest.csv"); ap.add_argument("--out", default="runs/cv1")
ap.add_argument("--folds", type=int, nargs="*", default=None)
ap.add_argument("--img-size", type=int, default=256); ap.add_argument("--batch", type=int, default=16)
ap.add_argument("--epochs-warm", type=int, default=3); ap.add_argument("--epochs-ft", type=int, default=25)
ap.add_argument("--mc-samples", type=int, default=16); ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--pop", type=int, default=20); ap.add_argument("--iters", type=int, default=25)
ap.add_argument("--no-imagenet", action="store_true"); ap.add_argument("--retrain", action="store_true")
ap.add_argument("--aggregate-only", action="store_true")
a = ap.parse_args()
KEYS = ["acc", "bal_acc", "f1_fake", "recall_fake", "specificity", "auc", "ece"]


def aggregate(out):
    folds = sorted(d for d in os.listdir(out) if d.startswith("fold") and os.path.exists(f"{out}/{d}/ablation_test.csv"))
    if not folds: print("no finished folds in", out); return
    print(f"\n================ aggregate over {len(folds)} fold(s): {folds} ================")
    for level, fname in [("IMAGE level", "ablation_test.csv"), ("SAMPLE level (all shots of a sample fused)", "ablation_sample_level.csv")]:
        tabs = [pd.read_csv(f"{out}/{d}/{fname}", index_col=0) for d in folds if os.path.exists(f"{out}/{d}/{fname}")]
        if not tabs: continue
        cat = pd.concat(tabs); g = cat.groupby(level=0, sort=False)[KEYS]
        mean, std = g.mean(), g.std(ddof=0)
        disp = mean.round(3).astype(str) + " ± " + std.round(3).astype(str)
        print(f"\n--- {level}: mean ± std over folds (test folds) ---"); print(disp.to_string())
        mean.to_csv(f"{out}/summary_{'image' if 'IMAGE' in level else 'sample'}_mean.csv"); disp.to_csv(f"{out}/summary_{'image' if 'IMAGE' in level else 'sample'}.csv")
    parts = []
    for d in folds:
        f = f"{out}/{d}/preds_test.csv"
        if not os.path.exists(f): continue
        q = pd.read_csv(f); th = json.load(open(f"{out}/{d}/thresholds.json"))
        for m in th:                                   # shift logits so each method's tuned threshold becomes 0.5
            pm = q[m].clip(1e-6, 1 - 1e-6); t = min(max(th[m], 1e-6), 1 - 1e-6)
            q[m] = 1 / (1 + np.exp(-(np.log(pm / (1 - pm)) - np.log(t / (1 - t)))))
        parts.append(q)
    P = pd.concat(parts)
    methods = [c for c in P.columns if c not in ("group", "y", "fold")]
    rows = {}
    for m in methods:
        ps, ys = wb.aggregate_by_group(P[m].values, P.y.values, P.group.values)
        rows[m] = wb.metrics(ys, ps, 0.5)
    print("\n--- pooled out-of-fold, SAMPLE level, threshold 0.5 (every sample is tested exactly once) ---")
    print(pd.DataFrame(rows).T[["acc", "bal_acc", "f1_fake", "recall_fake", "specificity", "auc"]].round(3).to_string())
    pd.DataFrame(rows).T.to_csv(f"{out}/pooled_oof_sample_level.csv")


if not a.aggregate_only:
    man = pd.read_csv(a.manifest); K = int(man.fold.max()) + 1
    os.makedirs(a.out, exist_ok=True)
    X = cb.features(man.path.tolist(), cache=f"{a.out}/colour_features.npz")
    for k in (a.folds if a.folds is not None else range(K)):
        od = f"{a.out}/fold{k}"; os.makedirs(od, exist_ok=True)
        if a.retrain or not (os.path.exists(f"{od}/cache_val.npz") and os.path.exists(f"{od}/cache_test.npz")):
            cmd = [sys.executable, os.path.join(os.path.dirname(os.path.abspath(__file__)), "train_fold.py"),
                   "--manifest", a.manifest, "--out", od, "--fold", str(k), "--img-size", str(a.img_size), "--batch", str(a.batch),
                   "--epochs-warm", str(a.epochs_warm), "--epochs-ft", str(a.epochs_ft), "--mc-samples", str(a.mc_samples),
                   "--seed", str(a.seed)] + (["--no-imagenet"] if a.no_imagenet else [])
            subprocess.run(cmd, check=True)                       # fresh process per fold -> no TF memory build-up
        val, test = wb.Cache.load(f"{od}/cache_val.npz"), wb.Cache.load(f"{od}/cache_test.npz")
        kv = (k + 1) % K
        te = man[man.fold == k].reset_index(drop=True); tr = man[~man.fold.isin([k, kv])]
        p_col = cb.fit_predict(X[tr.index.values], tr.label.values, X[man.index[man.fold == k].values])
        assert (te.group.values == test.groups).all() and (te.label.values == test.y).all(), "cache/manifest order mismatch"
        table, info = wb.evaluate_all(val, test, pop=a.pop, iters=a.iters, seed=k,
                                      extra_rows={"colour-statistics random forest (baseline)": (p_col, 0.5)})
        table.to_csv(f"{od}/ablation_test.csv"); info["sample_table"].to_csv(f"{od}/ablation_sample_level.csv")
        pd.DataFrame({**{m: p for m, p in info["probs"].items()}, "group": test.groups, "y": test.y, "fold": k}).to_csv(f"{od}/preds_test.csv", index=False)
        json.dump(info["thr"], open(f"{od}/thresholds.json", "w"))
        json.dump({q: (v.tolist() if hasattr(v, "tolist") else v) for q, v in info["params"].items()}, open(f"{od}/wombat_params.json", "w"), indent=1)
        print(f"\n=== fold {k} (test: {len(te)} images, {te.group.nunique()} samples) ===")
        print(table[["acc", "f1_fake", "recall_fake", "specificity", "auc"]].round(3).to_string())
aggregate(a.out)


In [ ]:
print('wrote /content/saffron-authenticity/scripts/run_cv.py')

In [ ]:
%%writefile /content/saffron-authenticity/scripts/train_fold.py
"""Train ONE fold in its own process (keeps TensorFlow memory from piling up across folds). Called by run_cv.py."""
import _path  # noqa
import argparse
import pandas as pd
import saffron_cnn_gnn as sg

ap = argparse.ArgumentParser()
ap.add_argument("--manifest", default="data/manifest.csv"); ap.add_argument("--out", required=True); ap.add_argument("--fold", type=int, required=True)
ap.add_argument("--img-size", type=int, default=256); ap.add_argument("--batch", type=int, default=16)
ap.add_argument("--epochs-warm", type=int, default=3); ap.add_argument("--epochs-ft", type=int, default=25)
ap.add_argument("--mc-samples", type=int, default=16); ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--no-imagenet", action="store_true")
a = ap.parse_args()
man = pd.read_csv(a.manifest); K = int(man.fold.max()) + 1
cfg = sg.Config(img_size=a.img_size, batch=a.batch, epochs_warm=a.epochs_warm, epochs_ft=a.epochs_ft,
                mc_samples=a.mc_samples, seed=a.seed, imagenet=not a.no_imagenet)
sg.run_fold(man, a.fold, K, cfg, a.out)
print("FOLD", a.fold, "DONE")


In [ ]:
print('wrote /content/saffron-authenticity/scripts/train_fold.py')

In [ ]:
%%writefile /content/saffron-authenticity/scripts/train.py
"""Train the CNN+GNN+HAFM model, then cache MC-dropout outputs for val and test.
   python scripts/train.py --data data --out runs/run1
"""
import _path  # noqa
import argparse, os, json, dataclasses
import pandas as pd, keras
import saffron_cnn_gnn as sg

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data"); ap.add_argument("--out", default="runs/run1")
ap.add_argument("--img-size", type=int, default=256); ap.add_argument("--batch", type=int, default=16)
ap.add_argument("--epochs-warm", type=int, default=3); ap.add_argument("--epochs-ft", type=int, default=25)
ap.add_argument("--mc-samples", type=int, default=16); ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--no-imagenet", action="store_true", help="random init (only for quick tests)")
a = ap.parse_args()
os.makedirs(a.out, exist_ok=True)
cfg = sg.Config(img_size=a.img_size, batch=a.batch, epochs_warm=a.epochs_warm, epochs_ft=a.epochs_ft,
                mc_samples=a.mc_samples, seed=a.seed, imagenet=not a.no_imagenet)
json.dump(dataclasses.asdict(cfg), open(f"{a.out}/config.json", "w"), indent=1)
keras.utils.set_random_seed(cfg.seed)
train_ds, val_ds, val_raw, test_raw = sg.make_datasets(a.data, cfg)
M = sg.build_models(cfg)
h1, h2 = sg.train(M, train_ds, val_ds, cfg, ckpt=f"{a.out}/best.weights.h5")
M["model"].save_weights(f"{a.out}/final.weights.h5")
pd.DataFrame(h2.history).to_csv(f"{a.out}/history_finetune.csv", index=False)
sg.extract_cache(M, val_raw, cfg).save(f"{a.out}/cache_val.npz")
sg.extract_cache(M, test_raw, cfg).save(f"{a.out}/cache_test.npz")
print("DONE ->", a.out)


In [ ]:
print('wrote /content/saffron-authenticity/scripts/train.py')

In [ ]:
%%writefile /content/saffron-authenticity/scripts/decode.py
"""Wombat-OA + weighted beam decoding on the cached outputs (no TensorFlow / GPU needed).
   python scripts/decode.py --out runs/run1
"""
import _path  # noqa
import argparse, json, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import wombat_beam as wb

ap = argparse.ArgumentParser()
ap.add_argument("--out", default="runs/run1"); ap.add_argument("--pop", type=int, default=20)
ap.add_argument("--iters", type=int, default=25); ap.add_argument("--seeds", type=int, default=10)
a = ap.parse_args()
val, test = wb.Cache.load(f"{a.out}/cache_val.npz"), wb.Cache.load(f"{a.out}/cache_test.npz")
N, V, H = test.shape
print(f"val {val.shape}  test {test.shape}  fake fraction test={test.y.mean():.2f}")

table, info = wb.evaluate_all(val, test, pop=a.pop, iters=a.iters, seed=0)
pd.set_option("display.width", 220, "display.max_colwidth", 70)
cols = ["acc", "bal_acc", "f1_fake", "recall_fake", "specificity", "auc", "ece", "nll", "fn", "fp"]
print(table[cols].round(4)); table.to_csv(f"{a.out}/ablation_test.csv")
json.dump({k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in info["params"].items()},
          open(f"{a.out}/wombat_params.json", "w"), indent=1)

cmp = wb.compare_optimizers(val, seeds=range(a.seeds), pop=a.pop, iters=a.iters)
print("\nbest val fitness  Wombat-OA %.4f+-%.4f | Random %.4f+-%.4f | Wilcoxon p=%.4g" %
      (cmp["wombat"].mean(), cmp["wombat"].std(), cmp["random"].mean(), cmp["random"].std(), cmp["wilcoxon_p"]))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for h in cmp["hist_wombat"]: ax[0].plot(np.linspace(0, 1, len(h)), h, c="tab:green", alpha=.35)
for h in cmp["hist_random"]: ax[0].plot(np.linspace(0, 1, len(h)), h, c="tab:gray", alpha=.35)
ax[0].set_title("convergence (green=Wombat-OA, gray=random)"); ax[0].set_xlabel("fraction of budget"); ax[0].set_ylabel("best val fitness")
ax[1].boxplot([cmp["wombat"], cmp["random"]], tick_labels=["Wombat-OA", "Random"]); ax[1].set_title(f"final fitness, {a.seeds} seeds")
plt.tight_layout(); plt.savefig(f"{a.out}/optimizer_comparison.png", dpi=200)

P = info["params"]; p_prop = wb.decode(test, P); p_base = wb.b_tta_mean_logit(test, "hafm")
print("\nbootstrap 95% CI on TEST")
for name, p, thr in [("proposed", p_prop, P["theta"]), ("TTA mean-logit", p_base, 0.5)]:
    for k in ("acc", "f1_fake", "auc"):
        print(f"  {name:16s}{k:8s}{wb.metrics(test.y, p, thr)[k]:.4f}  CI {tuple(round(x, 4) for x in wb.bootstrap_ci(test.y, p, thr, k))}")
print("DONE ->", a.out)


In [ ]:
print('wrote /content/saffron-authenticity/scripts/decode.py')

In [ ]:
%%writefile /content/saffron-authenticity/scripts/check_dataset.py
"""Validate the dataset BEFORE training: structure, readable images, class balance, size stats, cross-split duplicates."""
import _path  # noqa
import argparse, os, hashlib, collections, sys
from PIL import Image

ap = argparse.ArgumentParser(); ap.add_argument("--data", default="data"); a = ap.parse_args()
EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
ok, seen, dups = True, {}, []
counts = {}
for s in ("train", "val", "test"):
    for c in ("real", "fake"):
        d = os.path.join(a.data, s, c)
        if not os.path.isdir(d):
            print(f"[MISSING] {d}"); ok = False; continue
        files = [f for f in os.listdir(d) if f.lower().endswith(EXT)]
        counts[(s, c)] = len(files); bad = 0; sizes = collections.Counter()
        for f in files:
            p = os.path.join(d, f)
            try:
                with Image.open(p) as im:
                    im.verify()
                with Image.open(p) as im:
                    sizes[im.size] += 1
            except Exception as e:
                bad += 1; print(f"[CORRUPT] {p}: {e}")
            h = hashlib.md5(open(p, "rb").read()).hexdigest()
            if h in seen and seen[h][0] != s: dups.append((seen[h], (s, c, f)))
            seen.setdefault(h, (s, c, f))
        top = ", ".join(f"{w}x{h}:{n}" for (w, h), n in sizes.most_common(2))
        print(f"{s:5s} {c:5s} n={len(files):5d} corrupt={bad}  common sizes: {top}")
        ok &= (len(files) > 0 and bad == 0)
print("\nsplit sizes:", {s: sum(counts.get((s, c), 0) for c in ("real", "fake")) for s in ("train", "val", "test")})
for s in ("train", "val", "test"):
    r, f = counts.get((s, "real"), 0), counts.get((s, "fake"), 0)
    if r and f and max(r, f) / min(r, f) > 1.5: print(f"[WARN] {s} is imbalanced (real={r}, fake={f}) -> class weights handle train; use balanced metrics")
if dups: print(f"[FAIL] {len(dups)} exact duplicate(s) across splits (leakage), e.g. {dups[0]}"); ok = False
else: print("no exact cross-split duplicates")
print("\nDATASET OK" if ok else "\nDATASET HAS PROBLEMS"); sys.exit(0 if ok else 1)


In [ ]:
print('wrote /content/saffron-authenticity/scripts/check_dataset.py')

In [ ]:
%%writefile /content/saffron-authenticity/scripts/split_dataset.py
"""Split  data/raw/{real,fake}/*  ->  data/{train,val,test}/{real,fake}/*   (stratified).

Leakage warning: if several photos come from the SAME physical saffron sample / lot / session, they must land in the
same split.  Encode the sample id in the file name and pass --group-regex with ONE capture group, e.g.
    --group-regex "^(lot\\d+)_"         for names like  lot17_img003.jpg
Without --group-regex every image is treated as its own group (fine only if images are truly independent).
"""
import _path  # noqa
import argparse, os, re, shutil, random, collections

ap = argparse.ArgumentParser()
ap.add_argument("--src", default="data/raw"); ap.add_argument("--dst", default="data")
ap.add_argument("--val", type=float, default=0.15); ap.add_argument("--test", type=float, default=0.15)
ap.add_argument("--group-regex", default=None); ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--move", action="store_true", help="move instead of copy")
a = ap.parse_args()
rng = random.Random(a.seed)
EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
for cls in ("real", "fake"):
    d = os.path.join(a.src, cls)
    files = sorted(f for f in os.listdir(d) if f.lower().endswith(EXT))
    groups = collections.defaultdict(list)
    for f in files:
        m = re.search(a.group_regex, f) if a.group_regex else None
        groups[m.group(1) if m else f].append(f)
    keys = list(groups); rng.shuffle(keys)
    n = len(files); tgt = {"test": a.test * n, "val": a.val * n}; cnt = collections.Counter(); out = {}
    for k in keys:                                        # fill test, then val, remainder -> train
        s = "test" if cnt["test"] < tgt["test"] else "val" if cnt["val"] < tgt["val"] else "train"
        out[k] = s; cnt[s] += len(groups[k])
    for k, s in out.items():
        os.makedirs(os.path.join(a.dst, s, cls), exist_ok=True)
        for f in groups[k]:
            (shutil.move if a.move else shutil.copy2)(os.path.join(d, f), os.path.join(a.dst, s, cls, f))
    print(f"{cls}: {n} images, {len(groups)} groups ->", dict(cnt))


In [ ]:
print('wrote /content/saffron-authenticity/scripts/split_dataset.py')

In [ ]:
%%writefile /content/saffron-authenticity/scripts/make_loco_manifests.py
"""Leave-one-camera-out (LOCO) manifests: train on ONE phone, validate on held-out
samples of the SAME phone, test on the OTHER phone.

Why: inside each camera your cross-validation was already perfect, but that does
not show the model works on a device it has never seen. LOCO is the strongest
cross-device test your data allows (both phones contain real and fake samples).

   python scripts/make_loco_manifests.py --manifest data/manifest.csv
   python scripts/run_cv.py --manifest data/manifest_loco_<A>_to_<B>.csv --out runs/loco_<A>_to_<B> --folds 0 --img-size 256

Each output manifest re-uses the existing runner by encoding roles in the `fold` column:
   fold 0 = TEST (other camera) | fold 1 = VALIDATION (held-out groups of the training camera) | fold 2 = TRAIN (rest of the training camera)
Images with camera 'unknown' are excluded (they are all real and cannot form a two-class test).
Everything stays group-safe: a sample/group never appears in two roles.
"""
import _path  # noqa
import argparse, os, re
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

ap = argparse.ArgumentParser()
ap.add_argument("--manifest", default="data/manifest.csv")
ap.add_argument("--out-dir", default=None)
ap.add_argument("--val-splits", type=int, default=4,
                help="1/N of the training camera's groups become validation")
ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--min-val-groups-per-class", type=int, default=2)
a = ap.parse_args()

man = pd.read_csv(a.manifest)
if "camera" not in man:
    raise SystemExit("manifest has no 'camera' column: re-run scripts/prepare_dataset.py "
                     "(it reads EXIF camera models)")

out_dir = a.out_dir or os.path.dirname(os.path.abspath(a.manifest))
man["camera"] = man["camera"].astype(str)
known = man[man.camera != "unknown"]
print(f"excluded {len(man) - len(known)} image(s) with unknown camera "
      f"(classes: { {k: int(v) for k, v in man[man.camera == 'unknown'].cls.value_counts().items()} })")

usable = [c for c, g in known.groupby("camera")
          if g.label.nunique() == 2 and g.group.nunique() >= 6]
print("cameras with both classes and >= 6 groups:", usable)
if len(usable) < 2:
    raise SystemExit("need at least two usable cameras for leave-one-camera-out")


def sanitize(s):
    return re.sub(r"[^A-Za-z0-9]+", "", s)


def split_val(tr):
    """Split training-camera data into train / validation, ensuring both classes
    have >= N groups in the validation part."""
    for seed in range(a.seed, a.seed + 50):
        sg = StratifiedGroupKFold(a.val_splits, shuffle=True, random_state=seed)
        i_tr, i_val = next(iter(sg.split(tr, tr.label, tr.group)))
        v, t = tr.iloc[i_val], tr.iloc[i_tr]
        gv = v.groupby("label").group.nunique()
        if (len(gv) == 2
                and gv.min() >= a.min_val_groups_per_class
                and t.label.nunique() == 2):
            return t, v, seed
    raise SystemExit("could not build a validation split with enough groups "
                     "of both classes")


for tr_cam in usable:
    for te_cam in usable:
        if tr_cam == te_cam:
            continue
        tr_all = known[known.camera == tr_cam]
        te = known[known.camera == te_cam]
        tr, va, seed_used = split_val(tr_all)
        full = pd.concat([
            te.assign(fold=0),
            va.assign(fold=1),
            tr.assign(fold=2),
        ], ignore_index=True)

        # safety check: no group appears in more than one role
        g_roles = full.groupby("group").fold.nunique()
        assert g_roles.max() == 1, "group appears in more than one role"

        name = f"manifest_loco_{sanitize(tr_cam)}_to_{sanitize(te_cam)}.csv"
        full.to_csv(os.path.join(out_dir, name), index=False)
        print(f"\n=== train on {tr_cam}  ->  test on {te_cam}   "
              f"(wrote {name}, val split seed {seed_used}) ===")
        for role, df in [("TRAIN", tr), ("VAL", va), ("TEST", te)]:
            ft = {k: int(v) for k, v in df[df.label == 1].type.value_counts().items()}
            print(f"  {role:5s} {len(df):4d} images / {df.group.nunique():3d} samples "
                  f"| real {int((df.label == 0).sum()):3d}  "
                  f"fake {int((df.label == 1).sum()):3d} | fake types {ft}")

        # warn about unseen fake types in the test camera
        seen = set(pd.concat([tr, va])[lambda d: d.label == 1].type)
        unseen = sorted(set(te[te.label == 1].type) - seen)
        if unseen:
            print(f"  [NOTE] fake type(s) {unseen} appear ONLY in the test "
                  f"camera: this direction mixes a device shift with an "
                  f"unseen-adulterant shift")

        print(f"  run: python scripts/run_cv.py "
              f"--manifest {os.path.join(out_dir, name)} "
              f"--out runs/loco_{sanitize(tr_cam)}_to_{sanitize(te_cam)} "
              f"--folds 0")


In [ ]:
print('wrote /content/saffron-authenticity/scripts/make_loco_manifests.py')

In [ ]:
%%writefile /content/saffron-authenticity/scripts/stress_test.py
"""Stress test: re-evaluate the SAVED fold models on degraded test images
(no retraining, no retuning).

When every method scores 100% on clean images, nothing can be compared.
Degrading only the TEST images (blur, JPEG, noise, low resolution, grayscale,
hue shift, darkening) shows (a) what the network relies on (colour? fine
texture?) and (b) whether log-probability fusion, the weighted beam and the
Wombat-tuned parameters are more robust than a plain average.

   python scripts/stress_test.py --manifest data/manifest.csv --run runs/cv1
   python scripts/stress_test.py --manifest data/manifest.csv --run runs/cv1 --folds 0 1 --perturb clean blur jpeg

Needs per fold: config.json, final.weights.h5 (or best.weights.h5),
wombat_params.json, thresholds.json (all written by run_cv.py; weights are
only on the Colab disk unless you set SAVE_WEIGHTS_TO_DRIVE). Parameters tuned
on CLEAN validation data are reused unchanged.

The colour-statistics baseline is not re-evaluated here (it would need its own
feature extraction on degraded images).

Outputs: <run>/stress_results.csv and printed tables (sample level, one
decision per physical sample, pooled over folds).
"""
import _path  # noqa
import argparse, dataclasses, json, os
import numpy as np, pandas as pd
import tensorflow as tf
import saffron_cnn_gnn as sg
import wombat_beam as wb

ap = argparse.ArgumentParser()
ap.add_argument("--manifest", default="data/manifest.csv")
ap.add_argument("--run", required=True)
ap.add_argument("--folds", type=int, nargs="*", default=None)
ap.add_argument("--perturb", nargs="*",
                default=["clean", "blur", "jpeg", "noise", "lowres", "gray",
                         "hue", "dark"])
ap.add_argument("--batch", type=int, default=8)
a = ap.parse_args()


# ---------------------------------------------------------------------------
# Perturbation functions (applied to batches of 0-255 float32 images)
# ---------------------------------------------------------------------------
def _gauss_kernel(sigma=2.0, size=9):
    x = np.arange(size) - size // 2
    k = np.exp(-(x ** 2) / (2 * sigma ** 2))
    k /= k.sum()
    k2 = np.outer(k, k).astype("float32")
    return tf.constant(np.tile(k2[:, :, None, None], (1, 1, 3, 1)))


GK = _gauss_kernel()

PERTURB = {
    "clean":  None,
    "blur":   lambda x: tf.nn.depthwise_conv2d(x, GK, [1, 1, 1, 1], "SAME"),
    "jpeg":   lambda x: tf.map_fn(
        lambda im: tf.image.adjust_jpeg_quality(im / 255.0, 20) * 255.0, x),
    "noise":  lambda x: tf.clip_by_value(
        x + tf.random.normal(tf.shape(x), 0.0, 12.0, seed=0), 0.0, 255.0),
    "lowres": lambda x: tf.image.resize(
        tf.image.resize(x, (x.shape[1] // 4, x.shape[2] // 4)),
        (x.shape[1], x.shape[2])),
    "gray":   lambda x: tf.tile(
        tf.image.rgb_to_grayscale(x), [1, 1, 1, 3]),
    "hue":    lambda x: tf.clip_by_value(
        tf.image.adjust_hue(x / 255.0, 0.08) * 255.0, 0.0, 255.0),
    "dark":   lambda x: x * 0.6,
}


# ---------------------------------------------------------------------------
# Dataset builder (deterministic, no augmentation)
# ---------------------------------------------------------------------------
def dataset(df, S, batch, fn):
    ds = tf.data.Dataset.from_tensor_slices(
        (df.path.values.astype(str), df.label.values.astype("float32")))
    ds = ds.map(lambda p, l: (sg._load_image(p, S, False), l[None]),
                num_parallel_calls=tf.data.AUTOTUNE).batch(batch)
    if fn is not None:
        ds = ds.map(lambda x, y: (fn(x), y))
    return ds


# ---------------------------------------------------------------------------
# Method builder (mirrors the ablation table rows)
# ---------------------------------------------------------------------------
def methods_for(cache, wparams):
    H = len(cache.head_names)
    V = len(cache.view_names)
    hafm = cache.head_names[-1]

    rows = {f"single view, {h}": (wb.b_single(cache, h), 0.5)
            for h in cache.head_names}
    rows["late fusion (prob avg), identity view"] = \
        (wb.b_late_fusion_identity(cache), 0.5)
    rows["TTA mean-logit, hafm"] = (wb.b_tta_mean_logit(cache, hafm), 0.5)
    rows["TTA top-k confidence (old), hafm"] = \
        (wb.b_tta_topk_conf(cache, hafm), 0.5)
    rows["log-prob fusion, equal weights, no beam"] = \
        (wb.b_logprob_fusion_no_beam(cache), 0.5)

    p0 = wb.unpack(wb.default_params(H, V), H, V)
    rows["weighted beam + log-prob fusion (untuned)"] = \
        (wb.decode(cache, p0), p0["theta"])

    pw = dict(
        a=np.array(wparams["a"]),
        tau=np.array(wparams["tau"]),
        beta=wparams["beta"],
        gamma=wparams["gamma"],
        alpha=wparams["alpha"],
        beam=int(wparams["beam"]),
        depth=int(wparams["depth"]),
        kappa=wparams["kappa"],
        theta=wparams["theta"],
    )
    rows["PROPOSED: Wombat-OA + weighted beam + log-prob fusion"] = \
        (wb.decode(cache, pw), pw["theta"])

    return rows


# ---------------------------------------------------------------------------
# Main loop
# ---------------------------------------------------------------------------
man = pd.read_csv(a.manifest)
K = int(man.fold.max()) + 1
folds = a.folds if a.folds is not None else list(range(K))

store = {}  # (perturb, method) -> list of (p_shifted, y, group)

for k in folds:
    fd = f"{a.run}/fold{k}"
    w = next((f"{fd}/{n}" for n in ("final.weights.h5", "best.weights.h5")
              if os.path.exists(f"{fd}/{n}")), None)
    if w is None or not os.path.exists(f"{fd}/wombat_params.json"):
        print(f"fold {k}: skipped (no saved weights / params in {fd})")
        continue

    raw = json.load(open(f"{fd}/config.json"))
    cfg = sg.Config(**{f.name: raw[f.name]
                       for f in dataclasses.fields(sg.Config) if f.name in raw})
    cfg.imagenet = False  # weights come from the file

    M = sg.build_models(cfg)
    M["model"].load_weights(w)
    wparams = json.load(open(f"{fd}/wombat_params.json"))
    thr = json.load(open(f"{fd}/thresholds.json"))

    test_df = man[man.fold == k].reset_index(drop=True)
    print(f"\nfold {k}: {len(test_df)} test images / "
          f"{test_df.group.nunique()} samples, weights {os.path.basename(w)}")

    for pert in a.perturb:
        cache = sg.extract_cache(
            M, dataset(test_df, cfg.img_size, a.batch, PERTURB[pert]),
            cfg, groups=test_df.group.values)
        for name, (p, t) in methods_for(cache, wparams).items():
            t = min(max(t, 1e-6), 1 - 1e-6)
            pc = np.clip(p, 1e-6, 1 - 1e-6)
            # shift so each method's own threshold becomes 0.5
            ps = 1 / (1 + np.exp(-(np.log(pc / (1 - pc))
                                   - np.log(t / (1 - t)))))
            store.setdefault((pert, name), []).append(
                (ps, cache.y, cache.groups))
        print(f"   {pert:7s} done")
    tf.keras.backend.clear_session()

if not store:
    raise SystemExit("nothing evaluated: no fold had saved weights")

# ---------------------------------------------------------------------------
# Aggregate and print tables
# ---------------------------------------------------------------------------
rows = []
for (pert, name), parts in store.items():
    p = np.concatenate([x[0] for x in parts])
    y = np.concatenate([x[1] for x in parts])
    g = np.concatenate([x[2] for x in parts])
    ps, ys = wb.aggregate_by_group(p, y, g)
    for level, (pp, yy) in {"image": (p, y), "sample": (ps, ys)}.items():
        r = wb.metrics(yy, pp, 0.5)
        rows.append(dict(
            perturbation=pert, method=name, level=level, n=len(yy),
            acc=r["acc"], f1_fake=r["f1_fake"], recall_fake=r["recall_fake"],
            specificity=r["specificity"], auc=r["auc"]))

R = pd.DataFrame(rows)
R.to_csv(f"{a.run}/stress_results.csv", index=False)

pd.set_option("display.width", 250, "display.max_columns", 30)
for level in ("sample", "image"):
    for metric in ("acc", "auc"):
        T = R[R.level == level].pivot(
            index="method", columns="perturbation", values=metric)
        T = T[[c for c in a.perturb if c in set(R.perturbation)]]
        print(f"\n=== {metric.upper()} by perturbation, {level.upper()} level, "
              f"pooled over folds {folds} (rows = methods) ===\n"
              f"{T.round(3).to_string()}")

print(f"\nsaved {a.run}/stress_results.csv")


In [ ]:
print('wrote /content/saffron-authenticity/scripts/stress_test.py')

In [ ]:
%%writefile /content/saffron-authenticity/requirements.txt
# Python 3.10 - 3.12.   For GPU on Linux use:  pip install "tensorflow[and-cuda]>=2.16"  instead of the plain tensorflow line.
tensorflow>=2.16
keras>=3.3
numpy>=1.26
scipy>=1.11
scikit-learn>=1.3
pandas>=2.0
matplotlib>=3.8
pillow>=10.0
pytest>=8.0
nbformat>=5.9


In [ ]:
print('wrote /content/saffron-authenticity/requirements.txt')

In [ ]:
%%writefile /content/saffron-authenticity/tests/conftest.py
import os, sys
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
sys.path.insert(0, os.path.join(ROOT, "src"))


In [ ]:
print('wrote /content/saffron-authenticity/tests/conftest.py')

In [ ]:
%%writefile /content/saffron-authenticity/tests/test_dataset_prep.py
"""prepare_dataset.py on a miniature copy of the real naming scheme (Training/Testing share samples, one duplicate photo)."""
import os, subprocess, sys, zipfile
import numpy as np, pandas as pd
from PIL import Image

ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))


def _img(path, seed):
    rng = np.random.default_rng(seed)
    Image.fromarray(rng.integers(0, 255, (40 + seed % 7, 50, 3), dtype=np.uint8)).save(path)


def test_prepare_dataset_is_leak_free(tmp_path):
    src = tmp_path / "src"; seed = 0
    for cls, ids, typ in [("Fake", range(1, 9), ""), ("Fake", range(9, 12), " M"), ("Real", range(1, 13), "")]:
        for split, shots in [("Training", (1, 2, 3)), ("Testing", (4, 5))]:
            d = src / split / cls; d.mkdir(parents=True, exist_ok=True)
            for i in ids:
                for s in shots:
                    seed += 1; _img(d / f"{i}{typ}_{s}.jpg", seed)
    # a repeated photo under a DIFFERENT sample id (as in the real data): Real 2 shot 1 == Real 7 shot 1
    (src / "Training/Real/7_1.jpg").write_bytes((src / "Training/Real/2_1.jpg").read_bytes())
    r = subprocess.run([sys.executable, "scripts/prepare_dataset.py", "--src", str(src), "--data", str(tmp_path / "data"), "--folds", "4"],
                       cwd=ROOT, capture_output=True, text=True)
    assert r.returncode == 0, r.stdout + r.stderr
    assert "[LEAK]" in r.stdout and "[DUP]" in r.stdout
    m = pd.read_csv(tmp_path / "data" / "manifest.csv")
    assert len(m) == 8 * 5 + 3 * 5 + 12 * 5 - 1                                    # one duplicate file dropped
    assert m.groupby("group").fold.nunique().max() == 1                            # a group never spans two folds
    assert m[m["sample"] == "real---2"].group.iloc[0] == m[m["sample"] == "real---7"].group.iloc[0]   # duplicate merged
    assert set(m.type) == {"-", "M"} and m.fold.nunique() == 4
    assert m.groupby("fold").label.mean().between(0.2, 0.8).all()                    # stratified


def test_prepare_dataset_expands_zip_wildcards(tmp_path):
    archives = tmp_path / "archives"; archives.mkdir()
    for split, pattern, first_id in [("Testing", "Test_dataset", 1), ("Training", "train_dataset", 3)]:
        source = tmp_path / split
        for cls in ("Fake", "Real"):
            directory = source / split / cls
            directory.mkdir(parents=True)
            for sample_id in range(first_id, first_id + 2):
                _img(directory / f"{sample_id}_1.jpg", sample_id)
        with zipfile.ZipFile(archives / f"{pattern}-sample.zip", "w") as archive:
            for path in source.rglob("*.jpg"):
                archive.write(path, path.relative_to(source))

    r = subprocess.run(
        [sys.executable, "scripts/prepare_dataset.py", "--src",
         str(archives / "Test_dataset-*.zip"), str(archives / "train_dataset-*.zip"),
         "--data", str(tmp_path / "data"), "--folds", "2"],
        cwd=ROOT, capture_output=True, text=True,
    )
    assert r.returncode == 0, r.stdout + r.stderr
    assert "found 8 images" in r.stdout
    assert (tmp_path / "data" / "manifest.csv").exists()


def test_sample_level_aggregation():
    import wombat_beam as wb
    p = np.array([.9, .8, .2, .1, .3, .7]); y = np.array([1, 1, 0, 0, 0, 0]); g = np.array(["a", "a", "b", "b", "c", "c"])
    ps, ys = wb.aggregate_by_group(p, y, g)
    assert ys.tolist() == [1, 0, 0] and ps[0] > 0.8 and ps[1] < 0.2 and abs(ps[2] - 0.5) < 1e-6


In [ ]:
print('wrote /content/saffron-authenticity/tests/test_dataset_prep.py')

In [ ]:
%%writefile /content/saffron-authenticity/tests/test_decoder.py
"""Fast tests (seconds, no TensorFlow): optimizer, beam search, decoder, metrics."""
import itertools
import numpy as np
import wombat_beam as wb


def test_wombat_minimises_sphere_and_respects_bounds():
    f = lambda x: float(np.sum((x - 0.3) ** 2))
    lb, ub = -np.ones(6), np.ones(6) * 2
    x, fx, hist = wb.WombatOptimizer(f, lb, ub, pop_size=20, iters=40, seed=0).run()
    assert fx < 1e-2 and np.all(x >= lb) and np.all(x <= ub)
    assert np.all(np.diff(hist) <= 1e-12), "best fitness must never get worse (greedy acceptance)"


def test_beam_matches_exhaustive_search_when_beam_is_wide():
    rng = np.random.default_rng(0)
    N, V = 4, 5
    ell = -rng.random((N, V)); w = rng.uniform(.2, 1, (N, V))
    E = rng.normal(size=(N, V, 6)); E /= np.linalg.norm(E, axis=-1, keepdims=True)
    Sim = np.clip(np.einsum("nvd,nwd->nvw", E, E), 0, 1)
    gamma, alpha = 0.7, 0.6
    for L in (1, 2, 3, 5):
        S = wb._beam_one_class(ell, w, Sim, B=50, L=L, gamma=gamma, alpha=alpha)
        for n in range(N):
            best = {}
            for perm in itertools.permutations(range(V), L):
                s = ws = 0.0
                for i, v in enumerate(perm):
                    s += w[n, v] * ell[n, v] - gamma * max([Sim[n, v, u] for u in perm[:i]], default=0.0); ws += w[n, v]
                best[frozenset(perm)] = max(best.get(frozenset(perm), -1e9), s / ws ** alpha)
            assert np.allclose(np.sort(list(best.values()))[::-1], np.sort(S[n][np.isfinite(S[n])])[::-1])


def test_decode_outputs_valid_probabilities_and_beats_chance():
    c = wb.Cache.synthetic(N=400, seed=5, sep=0.8)
    N, V, H = c.shape
    p = wb.unpack(wb.default_params(H, V), H, V)
    prob = wb.decode(c, p)
    assert prob.shape == (N,) and np.all((prob >= 0) & (prob <= 1)) and np.isfinite(prob).all()
    assert wb.metrics(c.y, prob)["auc"] > 0.9


def test_wombat_tuning_improves_over_default_on_validation():
    val = wb.Cache.synthetic(N=400, seed=7, sep=0.35)
    N, V, H = val.shape
    fit = wb.make_fitness(val)
    x, f, _ = wb.tune(val, "wombat", pop=12, iters=8, seed=0)
    assert f <= fit(wb.default_params(H, V)) + 1e-9


def test_cache_roundtrip(tmp_path):
    c = wb.Cache.synthetic(N=50, seed=1); c.save(tmp_path / "c.npz"); d = wb.Cache.load(tmp_path / "c.npz")
    assert np.allclose(c.Z, d.Z) and d.head_names == c.head_names and d.view_names == c.view_names


def test_metrics_perfect_and_worst():
    y = np.array([0, 0, 1, 1])
    assert wb.metrics(y, np.array([.1, .2, .8, .9]))["acc"] == 1.0
    assert wb.metrics(y, np.array([.9, .8, .2, .1]))["acc"] == 0.0


In [ ]:
print('wrote /content/saffron-authenticity/tests/test_decoder.py')

In [ ]:
%%writefile /content/saffron-authenticity/tests/test_model_tf.py
"""Model-level checks (needs TensorFlow; ~1 min on CPU, random init, no download)."""
import numpy as np
import pytest
pytest.importorskip("tensorflow")
import saffron_cnn_gnn as sg


@pytest.fixture(scope="module")
def M():
    return sg.build_models(sg.Config(img_size=128, imagenet=False))


@pytest.fixture(scope="module")
def x():
    return np.random.default_rng(0).uniform(0, 255, (4, 128, 128, 3)).astype("float32")


def test_outputs_and_grid(M, x):
    out = M["model"](x, training=False)
    assert set(out) == set(sg.HEADS) and all(out[k].shape == (4, 1) for k in sg.HEADS)
    assert M["grid"] == (4, 4)                     # 128 / 32


def test_gnn_has_no_cross_sample_leakage(M, x):
    a, b = M["model"](x, training=False), M["model"](x[:1], training=False)
    assert all(np.allclose(a[k][0], b[k][0], atol=1e-4) for k in sg.HEADS)


def test_trunk_plus_heads_equals_model(M, x):
    f, a = M["trunk"](x, training=False), M["model"](x, training=False)
    assert all(np.allclose(M["heads"][k](f[k], training=False).numpy(), a[k].numpy(), atol=1e-4) for k in sg.HEADS)


def test_mc_dropout_only_when_training_true(M, x):
    f = M["trunk"](x, training=False); h = M["heads"]["hafm"]
    assert not np.allclose(h(f["hafm"], training=True).numpy(), h(f["hafm"], training=True).numpy())
    assert np.allclose(h(f["hafm"], training=False).numpy(), h(f["hafm"], training=False).numpy())


def test_hafm_attention_is_a_distribution(M, x):
    att = M["attn_model"](x).numpy()
    assert np.allclose(att.sum(1), 1.0, atol=1e-4) and (att >= 0).all()


def test_weights_roundtrip(M, x, tmp_path):
    p = str(tmp_path / "w.weights.h5"); M["model"].save_weights(p)
    M2 = sg.build_models(sg.Config(img_size=128, imagenet=False)); M2["model"].load_weights(p)
    a, b = M["model"](x, training=False), M2["model"](x, training=False)
    assert all(np.allclose(a[k], b[k], atol=1e-5) for k in sg.HEADS)


In [ ]:
print('wrote /content/saffron-authenticity/tests/test_model_tf.py')

In [ ]:
%%writefile /content/saffron-authenticity/tests/test_pipeline_e2e.py
"""End-to-end on a tiny synthetic dataset: check_dataset -> split_dataset -> train -> decode.  Slow (minutes on CPU)."""
import os, subprocess, sys
import pytest
pytest.importorskip("tensorflow")
import saffron_cnn_gnn as sg

ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
run = lambda *a, cwd=ROOT: subprocess.run([sys.executable, *a], cwd=cwd, capture_output=True, text=True, env={**os.environ, "TF_CPP_MIN_LOG_LEVEL": "3"})


@pytest.mark.slow
def test_full_pipeline(tmp_path):
    raw, data, out = tmp_path / "raw", tmp_path / "data", tmp_path / "run"
    sg._make_fake_dataset(str(tmp_path / "pre"), n=40, size=96)           # make images, then re-pool them as "raw"
    for cls in sg.CLASS_NAMES:
        os.makedirs(raw / cls)
        for i, f in enumerate(os.listdir(tmp_path / "pre" / "train" / cls)):
            os.replace(tmp_path / "pre" / "train" / cls / f, raw / cls / f"lot{i // 2}_{f}")
    r = run("scripts/split_dataset.py", "--src", str(raw), "--dst", str(data), "--group-regex", r"^(lot\d+)_"); assert r.returncode == 0, r.stderr
    r = run("scripts/check_dataset.py", "--data", str(data)); assert r.returncode == 0 and "DATASET OK" in r.stdout, r.stdout + r.stderr
    r = run("scripts/train.py", "--data", str(data), "--out", str(out), "--img-size", "128", "--batch", "8", "--epochs-warm", "1",
            "--epochs-ft", "1", "--mc-samples", "4", "--no-imagenet"); assert r.returncode == 0, r.stderr[-2000:]
    assert (out / "cache_val.npz").exists() and (out / "cache_test.npz").exists() and (out / "final.weights.h5").exists()
    r = run("scripts/decode.py", "--out", str(out), "--pop", "6", "--iters", "3", "--seeds", "2"); assert r.returncode == 0, r.stderr[-2000:]
    assert (out / "ablation_test.csv").exists() and (out / "optimizer_comparison.png").exists() and "DONE" in r.stdout


In [ ]:
print('wrote /content/saffron-authenticity/tests/test_pipeline_e2e.py')

In [ ]:
%%writefile /content/saffron-authenticity/tests/test_loco.py
"""prepare_dataset.py on a miniature copy with two cameras -> LOCO manifests are group-safe and two-class."""
import os, subprocess, sys
import numpy as np, pandas as pd

ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))


def test_loco_manifests_are_group_safe_and_two_class(tmp_path):
    rows = []
    for cam, n_real, n_fake in [("CAM-A", 12, 8), ("CAM-B", 8, 12)]:
        for k in range(n_real + n_fake):
            lab = int(k >= n_real)
            for s in range(3):
                rows.append(dict(
                    path=f"x_{cam}_{k}_{s}.jpg", label=lab,
                    cls="fake" if lab else "real",
                    type=("M" if k % 2 else "-") if lab else "-",
                    sample=f"{cam}{k}", group=f"{cam}-{k}", shot=s,
                    orig_split="train", fold=0, camera=cam))
    # one row with unknown camera (real-only, should be excluded)
    rows += [dict(path="u.jpg", label=0, cls="real", type="-", sample="u",
                  group="U", shot=1, orig_split="train", fold=0,
                  camera="unknown")]
    pd.DataFrame(rows).to_csv(tmp_path / "manifest.csv", index=False)

    r = subprocess.run(
        [sys.executable, "scripts/make_loco_manifests.py",
         "--manifest", str(tmp_path / "manifest.csv")],
        cwd=ROOT, capture_output=True, text=True)
    assert r.returncode == 0, r.stdout + r.stderr

    for a, b in [("CAMA", "CAMB"), ("CAMB", "CAMA")]:
        m = pd.read_csv(tmp_path / f"manifest_loco_{a}_to_{b}.csv")
        # all three roles present, unknown camera excluded
        assert set(m.fold) == {0, 1, 2} and "U" not in set(m.group)
        # no group in two roles
        assert m.groupby("group").fold.nunique().max() == 1
        # test camera is the OTHER one, train+val is the ONE
        assert set(m[m.fold == 0].camera) == {f"CAM-{b[-1]}"}
        assert set(m[m.fold != 0].camera) == {f"CAM-{a[-1]}"}
        # both classes in test, val and train
        for f in (0, 1, 2):
            assert m[m.fold == f].label.nunique() == 2


In [ ]:
print('wrote /content/saffron-authenticity/tests/test_loco.py')

In [ ]:
%%writefile /content/saffron-authenticity/pytest.ini
[pytest]
testpaths = tests
markers =
    slow: end-to-end tests that train a tiny model (a few minutes on CPU)


In [ ]:
print('wrote /content/saffron-authenticity/pytest.ini')

## § 5 · Data Preparation

In [ ]:
# § 5 — Copy zips from Drive → local /content disk, then run prepare_dataset.py
# Drive I/O is slow; we always train from /content local disk.
import os, glob, shutil, subprocess, sys, time

COLAB_ROOT   = globals().get("COLAB_ROOT",  "/content/saffron-authenticity")
LOCAL_DATA   = globals().get("LOCAL_DATA",  os.path.join(COLAB_ROOT, "data"))
DRIVE_DATA_DIR = globals().get("DRIVE_DATA_DIR", "")
N_FOLDS      = globals().get("N_FOLDS", 5)
SEED         = globals().get("SEED", 42)

os.makedirs(LOCAL_DATA, exist_ok=True)

# ── Subprocess output helper ───────────────────────────────────────────────────────────
import subprocess, sys
def run(cmd, cwd=COLAB_ROOT, env_extra=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2"}
    if env_extra: env.update(env_extra)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

# ── Locate source data ─────────────────────────────────────────────────────────────────
manifest_path = os.path.join(LOCAL_DATA, "manifest.csv")
if os.path.exists(manifest_path):
    print(f"manifest.csv already exists ({manifest_path}) — skipping data copy and prepare step.")
    print("Delete data/manifest.csv and re-run this cell to force a fresh preparation.")
else:
    # Find inputs: zips or folders in DRIVE_DATA_DIR
    zips = sorted(glob.glob(os.path.join(DRIVE_DATA_DIR, "*.zip")))
    folders = [d for d in glob.glob(os.path.join(DRIVE_DATA_DIR, "*"))
               if os.path.isdir(d) and os.path.basename(d).lower() in ("training","testing","train","test")]

    if len(zips) == 2:
        print(f"Found 2 zip files: {[os.path.basename(z) for z in zips]}")
        local_zips = []
        for z in zips:
            dst = os.path.join(LOCAL_DATA, os.path.basename(z))
            if not os.path.exists(dst):
                print(f"  copying {os.path.basename(z)} to local disk ...", end="", flush=True)
                shutil.copy2(z, dst); print(" done")
            else:
                print(f"  {os.path.basename(z)} already on local disk")
            local_zips.append(dst)
        src_args = local_zips
    elif len(zips) > 2:
        raise RuntimeError(
            f"Expected exactly 2 zip files in {DRIVE_DATA_DIR}, found {len(zips)}: {[os.path.basename(z) for z in zips]}\n"
            "Please keep only the Training zip and the Testing zip in that folder.")
    elif folders:
        print(f"No zips found; using folders: {[os.path.basename(f) for f in folders]}")
        src_args = folders
    else:
        try:
            from google.colab import files as _cfiles
            print("No zips found in Drive. Please upload the two dataset zips now:")
            uploaded = _cfiles.upload()
            src_args = []
            for fname in uploaded:
                dst = os.path.join(LOCAL_DATA, fname)
                with open(dst, "wb") as f: f.write(uploaded[fname])
                src_args.append(dst)
            if len(src_args) != 2:
                raise RuntimeError(f"Expected 2 zips, got {len(src_args)}: {src_args}")
        except (ModuleNotFoundError, RuntimeError) as e:
            raise RuntimeError(
                f"No dataset found in {DRIVE_DATA_DIR} and not in Colab.\n"
                f"Put Test_dataset-*.zip and train_dataset-*.zip in {DRIVE_DATA_DIR} and re-run.") from e

    print("\nRunning prepare_dataset.py ...")
    run([sys.executable, "scripts/prepare_dataset.py",
         "--src", *src_args,
         "--data", LOCAL_DATA,
         "--folds", str(N_FOLDS),
         "--seed", str(SEED)])

print("\n§5 preparation complete ✅")


In [ ]:
# § 5 — Display: fold table, montage of random real/fake images
import os, pandas as pd, matplotlib.pyplot as plt, matplotlib.image as mpimg
import numpy as np, collections

COLAB_ROOT   = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_DATA   = globals().get("LOCAL_DATA", os.path.join(COLAB_ROOT, "data"))
FIGURES_DIR  = globals().get("FIGURES_DIR", os.path.join(COLAB_ROOT, "runs","cv1","figures"))
os.makedirs(FIGURES_DIR, exist_ok=True)

man = pd.read_csv(os.path.join(LOCAL_DATA, "manifest.csv"))
print(f"Manifest: {len(man)} images  |  {man.group.nunique()} physical samples  |  "
      f"{(man.label==1).sum()} fake / {(man.label==0).sum()} real")

# Per-fold summary
rep = man.groupby("fold").agg(
    images=("path","size"), samples=("group","nunique"), fake_frac=("label","mean")
)
rep["types(fake)"] = [
    dict(collections.Counter(man[(man.fold==k)&(man.label==1)].type)) for k in rep.index
]
print("\nPer-fold summary (each fold is the TEST set once):")
print(rep.round(3).to_string())

# 2×5 montage
rng = np.random.default_rng(42)
fig, axes = plt.subplots(2, 5, figsize=(14, 5.5))
for col, (label, name) in enumerate([(0,"Real"),(1,"Fake")]):
    pool = man[man.label==label].path.values
    chosen = rng.choice(pool, min(5, len(pool)), replace=False)
    for i, p in enumerate(chosen):
        row = 0 if label==0 else 1
        ax = axes[row, i]
        try:
            img = mpimg.imread(p)
            ax.imshow(img); ax.axis("off")
            ax.set_title(f"{name} {i+1}", fontsize=8)
        except Exception as e:
            ax.text(0.5,0.5,str(e),ha="center",va="center",fontsize=6); ax.axis("off")
plt.suptitle("Sample images from the working copies", fontsize=11)
plt.tight_layout()
fn = os.path.join(FIGURES_DIR, "montage.png")
plt.savefig(fn, dpi=120, bbox_inches="tight"); plt.show()
print(f"Saved → {fn}")
print("§5 display complete ✅")


## § 5a · Dataset Audit

Checks for the known dataset problems described in `README.md § 0`:
- **Leak**: how many of the 57 original test samples also appear in the training folder (expected: all 57)
- **Duplicates**: exact-duplicate files
- **Image-size histogram**: shows whether image resolution is a shortcut cue (real photos tend to be larger)
- **Shortcut AUC**: group-CV AUC from (a) size/aspect/file-size features, (b) colour statistics, (c) both combined

⚠ **Red flag**: colour-stats AUC > 0.9 means trivial colour features explain most of the task — any deep model must beat this.


In [ ]:
# § 5a — Dataset audit: leak summary, duplicates, size histogram, shortcut AUC
import os, sys, subprocess, collections
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_DATA  = globals().get("LOCAL_DATA", os.path.join(COLAB_ROOT, "data"))
FIGURES_DIR = globals().get("FIGURES_DIR", os.path.join(COLAB_ROOT,"runs","cv1","figures"))
sys.path.insert(0, os.path.join(COLAB_ROOT, "src"))
sys.path.insert(0, os.path.join(COLAB_ROOT, "scripts"))
os.makedirs(FIGURES_DIR, exist_ok=True)

man = pd.read_csv(os.path.join(LOCAL_DATA, "manifest.csv"))

# ── Leak summary (re-compute from orig_split column) ────────────────────────────────────
tr_samp = set(man[man.orig_split=="train"].sample)
te_samp = set(man[man.orig_split=="test"].sample)
n_leaked = len(tr_samp & te_samp)
n_te     = len(te_samp)
print(f"[LEAK] {n_leaked}/{n_te} original test samples also in training (different shots)")
assert n_te > 0, "orig_split column missing test rows — re-run §5 with the original zips"

# Bar
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(["Only train", "Only test", "BOTH (leak)"],
            [len(tr_samp)-n_leaked, n_te-n_leaked, n_leaked],
            color=["steelblue","orange","crimson"])
axes[0].set_title("Sample leak: original split"); axes[0].set_ylabel("# samples")

# ── Image-size histogram ─────────────────────────────────────────────────────────────────
widths, heights = [], []
file_sizes = []
for _, row in man.iterrows():
    try:
        with Image.open(row.path) as im:
            w, h = im.size
        widths.append(w); heights.append(h)
        file_sizes.append(os.path.getsize(row.path))
    except Exception:
        widths.append(0); heights.append(0); file_sizes.append(0)
man = man.copy()
man["img_w"], man["img_h"] = widths, heights
man["img_area"] = np.array(widths) * np.array(heights)
man["file_bytes"] = file_sizes
man["aspect"] = np.array(widths) / np.maximum(np.array(heights), 1)

for label, name, c in [(0,"Real","steelblue"),(1,"Fake","crimson")]:
    sub = man[man.label==label]
    axes[1].hist(sub.img_w.clip(0,3000)/1000, bins=25, alpha=0.6, label=name, color=c)
axes[1].set_xlabel("Width (k px)"); axes[1].set_ylabel("Count")
axes[1].set_title("Image-width distribution by class"); axes[1].legend()

# ── Shortcut AUC: size/aspect/file-size and colour ─────────────────────────────────────
import color_baseline as cb

X_col = cb.features(man.path.tolist())
X_size = np.column_stack([man.img_w, man.img_h, man.img_area, man.aspect, man.file_bytes])

K = man.fold.max() + 1
results = {"size_feat": [], "colour": [], "both": []}
for k in range(K):
    kv = (k + 1) % K
    tr_idx = man[~man.fold.isin([k, kv])].index.values
    te_idx = man[man.fold == k].index.values
    ytr, yte = man.label.values[tr_idx], man.label.values[te_idx]
    for key, X in [("size_feat", X_size), ("colour", X_col), ("both", np.hstack([X_size, X_col]))]:
        p = cb.fit_predict(X[tr_idx], ytr, X[te_idx])
        results[key].append(roc_auc_score(yte, p) if len(np.unique(yte)) > 1 else float("nan"))

print(f"\nShortcut AUC (image-level, group-CV):")
for k, v in results.items():
    print(f"  {k:15s}  per-fold: {[round(x,3) for x in v]}  mean={np.nanmean(v):.3f}")

axes[2].boxplot([results["size_feat"], results["colour"], results["both"]],
                tick_labels=["Size/aspect", "Colour", "Both"])
axes[2].set_ylim(0.4, 1.05); axes[2].axhline(0.5, ls="--", c="grey")
axes[2].set_title("Shortcut AUC (group-CV)"); axes[2].set_ylabel("AUC")

plt.tight_layout()
fn = os.path.join(FIGURES_DIR, "dataset_audit.png")
plt.savefig(fn, dpi=120, bbox_inches="tight"); plt.show()
print(f"Saved → {fn}")

assert np.nanmean(results["colour"]) > 0.5,     "Colour AUC ≤ 0.5 — unexpected. Check that manifest labels are correct."
print(f"\nColour AUC {np.nanmean(results['colour']):.3f} — this is the bar every deep model must beat.")
print("§5a complete ✅")


## § 6 · Colour-Statistics Baseline (CPU)

Uses the same 5 folds as `run_cv.py` (test=fold k, val=fold k+1, train=rest) so the AUC is directly comparable to deep-model results.
The colour features are RGB mean/std + HSV mean/std on a 64×64 centre crop — 12 numbers per image.

**Why this matters**: colour alone reaches AUC ~0.94 on this dataset. Any deep model row in the final table that falls below this baseline is not learning anything beyond colour.


In [ ]:
# § 6 — Colour-statistics baseline under the same 5 folds as the deep model
import os, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_DATA  = globals().get("LOCAL_DATA", os.path.join(COLAB_ROOT, "data"))
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
FIGURES_DIR = globals().get("FIGURES_DIR", os.path.join(LOCAL_RUNS, "figures"))
sys.path.insert(0, os.path.join(COLAB_ROOT, "src"))
sys.path.insert(0, os.path.join(COLAB_ROOT, "scripts"))
os.makedirs(FIGURES_DIR, exist_ok=True)

import color_baseline as cb

man = pd.read_csv(os.path.join(LOCAL_DATA, "manifest.csv"))
cache_file = os.path.join(LOCAL_RUNS, "colour_features.npz")
X = cb.features(man.path.tolist(), cache=cache_file)
K = man.fold.max() + 1

fold_aucs, fold_accs = [], []
print(f"{'Fold':>5}  {'AUC':>6}  {'Acc':>6}")
for k in range(K):
    kv = (k + 1) % K
    tr_idx = man[~man.fold.isin([k, kv])].index.values
    te_idx = man[man.fold == k].index.values
    ytr, yte = man.label.values[tr_idx], man.label.values[te_idx]
    p = cb.fit_predict(X[tr_idx], ytr, X[te_idx])
    auc = roc_auc_score(yte, p) if len(np.unique(yte)) > 1 else float("nan")
    acc = accuracy_score(yte, p >= 0.5)
    fold_aucs.append(auc); fold_accs.append(acc)
    print(f"  {k:3d}    {auc:.4f}  {acc:.4f}")

print(f"  mean   {np.nanmean(fold_aucs):.4f}  {np.mean(fold_accs):.4f}")
print(f"  std    {np.nanstd(fold_aucs):.4f}  {np.std(fold_accs):.4f}")

fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))
ax.bar(range(K), fold_aucs, color="steelblue", alpha=0.8)
ax.axhline(np.nanmean(fold_aucs), ls="--", c="navy", label=f"mean {np.nanmean(fold_aucs):.3f}")
ax.set_xticks(range(K)); ax.set_xticklabels([f"Fold {k}" for k in range(K)])
ax.set_ylim(0, 1.05); ax.set_ylabel("AUC"); ax.set_title("Colour-statistics baseline AUC per fold")
ax.legend(); plt.tight_layout()
fn = os.path.join(FIGURES_DIR, "baseline_auc.png")
plt.savefig(fn, dpi=120, bbox_inches="tight"); plt.show(); print(f"Saved → {fn}")

print(f"\n⚠  The bar to beat: colour baseline mean AUC = {np.nanmean(fold_aucs):.3f}")
print("§6 complete ✅")


## § 7 · Smoke Test (optional, runs only when `SMOKE=True`)

Runs fold 0 with tiny settings and random weights. **Numbers here are NOT results** — this is a pipeline sanity check only.
Expected runtime: ~5–15 min on GPU, ~30–60 min on CPU.


In [ ]:
# § 7 — Smoke test: fold 0, tiny settings, no ImageNet weights
import os, sys, subprocess, time
SMOKE      = globals().get("SMOKE", False)
COLAB_ROOT = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_DATA = globals().get("LOCAL_DATA", os.path.join(COLAB_ROOT, "data"))
IMG_SIZE   = globals().get("IMG_SIZE", 128)
MC_SAMPLES = globals().get("MC_SAMPLES", 4)
POP        = globals().get("POP", 8)
ITERS      = globals().get("ITERS", 4)
SEED       = globals().get("SEED", 42)

def run(cmd, cwd=COLAB_ROOT, env_extra=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2"}
    if env_extra: env.update(env_extra)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

if not SMOKE:
    print("SMOKE=False — skipping smoke test. Set SMOKE=True in §2 to enable.")
else:
    print("Running smoke test (fold 0, no ImageNet, tiny settings) ...")
    t0 = time.time()
    run([sys.executable, "scripts/run_cv.py",
         "--manifest", os.path.join(LOCAL_DATA, "manifest.csv"),
         "--out", os.path.join(COLAB_ROOT, "runs", "smoke"),
         "--folds", "0",
         "--img-size", str(IMG_SIZE),
         "--epochs-warm", "1",
         "--epochs-ft", "1",
         "--mc-samples", str(MC_SAMPLES),
         "--no-imagenet",
         "--pop", str(POP),
         "--iters", str(ITERS),
         "--seed", str(SEED)])
    elapsed = time.time() - t0
    smoke_csv = os.path.join(COLAB_ROOT, "runs", "smoke", "fold0", "ablation_test.csv")
    assert os.path.exists(smoke_csv), f"Smoke test failed: {smoke_csv} not created"
    print(f"\n✅ Smoke test passed in {elapsed/60:.1f} min — pipeline is healthy.")
    print("Numbers above are NOT results (random weights, tiny settings).")
print("§7 complete ✅")


## § 8 · Resume Logic (re-run after a Colab disconnect)

In [ ]:
# § 8 — If Drive has a previous runs/cv1, rsync it to local disk before training.
# Re-running this cell on a fresh session restores all finished folds automatically.
import os, shutil, subprocess

COLAB_ROOT     = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_RUNS     = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
DRIVE_RUNS_DIR = globals().get("DRIVE_RUNS_DIR", "")

drive_cv1 = os.path.join(DRIVE_RUNS_DIR, "cv1")
os.makedirs(LOCAL_RUNS, exist_ok=True)

if os.path.isdir(drive_cv1):
    print(f"Found previous run in Drive: {drive_cv1} — syncing to local disk ...")
    # Try rsync first, fall back to shutil
    rsync_ok = subprocess.run(
        ["rsync", "-a", "--exclude=*.weights.h5", drive_cv1 + "/", LOCAL_RUNS + "/"],
        capture_output=True
    ).returncode == 0
    if rsync_ok:
        print("rsync complete ✅")
    else:
        print("rsync not available — using shutil.copytree fallback ...")
        # Copy only non-weight files
        for dp, dns, fns in os.walk(drive_cv1):
            rel = os.path.relpath(dp, drive_cv1)
            dst_dp = os.path.join(LOCAL_RUNS, rel)
            os.makedirs(dst_dp, exist_ok=True)
            for fn in fns:
                if fn.endswith(".weights.h5"): continue
                src_f = os.path.join(dp, fn)
                dst_f = os.path.join(dst_dp, fn)
                if not os.path.exists(dst_f):
                    shutil.copy2(src_f, dst_f)
        print("shutil copy complete ✅")

    # Report what folds are already done
    done = [d for d in os.listdir(LOCAL_RUNS)
            if d.startswith("fold") and os.path.exists(os.path.join(LOCAL_RUNS, d, "cache_test.npz"))]
    print(f"Folds already complete: {sorted(done)}")
else:
    print(f"No previous run found in {drive_cv1} — starting fresh.")
print("§8 complete ✅")


## § 9 · Full Cross-Validation

Trains all folds. Each fold spawns a fresh subprocess so TensorFlow GPU memory is released between folds.
Finished folds (both `cache_val.npz` and `cache_test.npz` present) are skipped automatically.
After each fold the results are synced to Drive (excluding weights unless `SAVE_WEIGHTS_TO_DRIVE=True`).

Training log: one line per epoch (Keras `verbose=2`). ETA is updated after each fold.


In [ ]:
# § 9 — Full cross-validation: train → cache → Wombat-OA decode, for every fold
import os, sys, subprocess, time, shutil

COLAB_ROOT            = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_DATA            = globals().get("LOCAL_DATA", os.path.join(COLAB_ROOT, "data"))
LOCAL_RUNS            = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
DRIVE_RUNS_DIR        = globals().get("DRIVE_RUNS_DIR", "")
N_FOLDS               = globals().get("N_FOLDS", 5)
IMG_SIZE              = globals().get("IMG_SIZE", 256)
BATCH                 = globals().get("BATCH", 16)
EPOCHS_WARM           = globals().get("EPOCHS_WARM", 3)
EPOCHS_FT             = globals().get("EPOCHS_FT", 25)
MC_SAMPLES            = globals().get("MC_SAMPLES", 16)
POP                   = globals().get("POP", 20)
ITERS                 = globals().get("ITERS", 25)
SEED                  = globals().get("SEED", 42)
SMOKE                 = globals().get("SMOKE", False)
SAVE_WEIGHTS_TO_DRIVE = globals().get("SAVE_WEIGHTS_TO_DRIVE", False)
ALLOW_CPU             = globals().get("ALLOW_CPU", False)

import pandas as pd
man = pd.read_csv(os.path.join(LOCAL_DATA, "manifest.csv"))
K   = int(man.fold.max()) + 1
folds_to_run = [0] if SMOKE else list(range(K))

os.makedirs(LOCAL_RUNS, exist_ok=True)
drive_cv1 = os.path.join(DRIVE_RUNS_DIR, "cv1")

def run(cmd, cwd=COLAB_ROOT, env_extra=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2"}
    if env_extra: env.update(env_extra)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

def sync_to_drive():
    if not DRIVE_RUNS_DIR:
        return
    os.makedirs(drive_cv1, exist_ok=True)
    exclude = [] if SAVE_WEIGHTS_TO_DRIVE else ["--exclude=*.weights.h5"]
    r = subprocess.run(
        ["rsync", "-a", *exclude, LOCAL_RUNS + "/", drive_cv1 + "/"],
        capture_output=True
    )
    if r.returncode != 0:
        # fallback: shutil
        for dp, _, fns in os.walk(LOCAL_RUNS):
            rel = os.path.relpath(dp, LOCAL_RUNS)
            ddp = os.path.join(drive_cv1, rel); os.makedirs(ddp, exist_ok=True)
            for fn in fns:
                if fn.endswith(".weights.h5") and not SAVE_WEIGHTS_TO_DRIVE: continue
                shutil.copy2(os.path.join(dp, fn), os.path.join(ddp, fn))
    try:
        os.sync()
    except AttributeError:
        pass  # Windows

fold_times = []
t_total_start = time.time()
for fold in folds_to_run:
    od = os.path.join(LOCAL_RUNS, f"fold{fold}")
    if (os.path.exists(os.path.join(od, "cache_val.npz")) and
            os.path.exists(os.path.join(od, "cache_test.npz"))):
        print(f"\n→ Fold {fold}: caches found, skipping training.")
        continue

    t0 = time.time()
    print(f"\n{'='*60}")
    print(f"FOLD {fold} / {K-1} — training starting")
    print(f"{'='*60}")
    cmd = [sys.executable, "scripts/train_fold.py",
           "--manifest", os.path.join(LOCAL_DATA, "manifest.csv"),
           "--out", od,
           "--fold", str(fold),
           "--img-size", str(IMG_SIZE),
           "--batch",    str(BATCH),
           "--epochs-warm", str(EPOCHS_WARM),
           "--epochs-ft",   str(EPOCHS_FT),
           "--mc-samples",  str(MC_SAMPLES),
           "--seed",        str(SEED)]
    if SMOKE:
        cmd.append("--no-imagenet")
    try:
        run(cmd)
    except subprocess.CalledProcessError as e:
        print(f"\n❌ Fold {fold} FAILED: {e}")
        print("Next action: check the error above. Common causes: OOM (reduce BATCH or IMG_SIZE), TF version mismatch.")
        raise

    elapsed = time.time() - t0
    fold_times.append(elapsed)
    mean_t = sum(fold_times) / len(fold_times)
    remaining = [f for f in folds_to_run
                 if not (os.path.exists(os.path.join(LOCAL_RUNS,f"fold{f}","cache_val.npz")) and
                         os.path.exists(os.path.join(LOCAL_RUNS,f"fold{f}","cache_test.npz")))]
    eta = mean_t * max(len(remaining) - 1, 0)
    print(f"\n✅ Fold {fold} complete in {elapsed/60:.1f} min  |  ETA remaining: {eta/60:.0f} min")

    # Wombat-OA decoding for this fold
    import color_baseline as cb
    import wombat_beam as wb
    import json, numpy as np

    val_c  = wb.Cache.load(os.path.join(od, "cache_val.npz"))
    test_c = wb.Cache.load(os.path.join(od, "cache_test.npz"))
    kv = (fold + 1) % K
    te_idx = man[man.fold == fold].index.values
    tr_idx = man[~man.fold.isin([fold, kv])].index.values
    X_col  = cb.features(man.path.tolist(), cache=os.path.join(LOCAL_RUNS, "colour_features.npz"))
    p_col  = cb.fit_predict(X_col[tr_idx], man.label.values[tr_idx], X_col[te_idx])
    assert (man.loc[te_idx, "group"].values == test_c.groups).all(), "cache/manifest order mismatch"
    assert (man.loc[te_idx, "label"].values == test_c.y).all(), "cache/manifest label mismatch"
    table, info = wb.evaluate_all(val_c, test_c, pop=POP, iters=ITERS, seed=fold,
                                  extra_rows={"colour-statistics random forest (baseline)": (p_col, 0.5)})
    table.to_csv(os.path.join(od, "ablation_test.csv"))
    info["sample_table"].to_csv(os.path.join(od, "ablation_sample_level.csv"))
    pd.DataFrame({
        **{m: p for m, p in info["probs"].items()},
        "group": test_c.groups, "y": test_c.y, "fold": fold
    }).to_csv(os.path.join(od, "preds_test.csv"), index=False)
    json.dump(info["thr"], open(os.path.join(od, "thresholds.json"), "w"))
    json.dump({q: (v.tolist() if hasattr(v,"tolist") else v) for q,v in info["params"].items()},
              open(os.path.join(od, "wombat_params.json"), "w"), indent=1)

    KEYS = ["acc","bal_acc","f1_fake","recall_fake","specificity","auc","ece"]
    print(f"\n--- Fold {fold} ablation (image level) ---")
    print(table[KEYS].round(3).to_string())

    print(f"\nSyncing fold {fold} to Drive ...")
    sync_to_drive()

total_elapsed = time.time() - t_total_start
print(f"\n{'='*60}")
print(f"All folds complete. Total time: {total_elapsed/60:.1f} min")
print(f"{'='*60}")
print("§9 complete ✅")


## § 10 · Aggregate Results & Summary Tables

In [ ]:
# § 10 — Aggregate results across folds; load summary tables; styled display + bar chart
import os, sys, subprocess, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from IPython.display import display

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_DATA  = globals().get("LOCAL_DATA", os.path.join(COLAB_ROOT, "data"))
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
FIGURES_DIR = globals().get("FIGURES_DIR", os.path.join(LOCAL_RUNS, "figures"))
os.makedirs(FIGURES_DIR, exist_ok=True)
sys.path.insert(0, os.path.join(COLAB_ROOT, "src"))
sys.path.insert(0, os.path.join(COLAB_ROOT, "scripts"))

def run(cmd, cwd=COLAB_ROOT, env_extra=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2"}
    if env_extra: env.update(env_extra)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)
    for line in proc.stdout: print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

# Re-run aggregate step to get fresh summary CSVs
run([sys.executable, "scripts/run_cv.py",
     "--manifest", os.path.join(LOCAL_DATA, "manifest.csv"),
     "--out", LOCAL_RUNS,
     "--aggregate-only"])

KEYS = ["acc","bal_acc","f1_fake","recall_fake","specificity","auc","ece"]
PROPOSED = "PROPOSED: Wombat-OA + weighted beam + log-prob fusion"
BASELINE = "colour-statistics random forest (baseline)"

for level, fname in [("IMAGE level", "summary_image.csv"),
                     ("SAMPLE level", "summary_sample.csv"),
                     ("Pooled OOF (SAMPLE)", "pooled_oof_sample_level.csv")]:
    fp = os.path.join(LOCAL_RUNS, fname)
    if not os.path.exists(fp):
        print(f"⚠ {fname} not found — run §9 first"); continue
    df = pd.read_csv(fp, index_col=0)
    # For summary files, values are strings "mean ± std"; for pooled they are floats
    is_float = df.dtypes.iloc[0] != object
    if is_float:
        cols = [c for c in KEYS if c in df.columns]
        styled = df[cols].style             .highlight_max(axis=0, color="lightgreen")             .format("{:.4f}")
    else:
        cols = [c for c in KEYS if c in df.columns]
        styled = df[cols].style
    print(f"\n=== {level} ===")
    display(styled)

# ── Grouped bar chart: mean AUC and F1(fake) with std bars ─────────────────────────────
mean_fp = os.path.join(LOCAL_RUNS, "summary_sample_mean.csv")
if os.path.exists(mean_fp):
    mean_df = pd.read_csv(mean_fp, index_col=0)
    # Compute std from per-fold data
    folds = sorted(d for d in os.listdir(LOCAL_RUNS)
                   if d.startswith("fold") and os.path.exists(os.path.join(LOCAL_RUNS, d, "ablation_sample_level.csv")))
    tabs = [pd.read_csv(os.path.join(LOCAL_RUNS, d, "ablation_sample_level.csv"), index_col=0)
            for d in folds]
    cat = pd.concat(tabs)
    std_df = cat.groupby(level=0, sort=False)[KEYS].std(ddof=0)

    methods = mean_df.index.tolist()
    x = np.arange(len(methods))
    fig, axes = plt.subplots(1, 2, figsize=(max(12, len(methods)*0.6), 5))
    colors = plt.cm.tab20(np.linspace(0, 1, len(methods)))
    for ai, col in enumerate(["auc", "f1_fake"]):
        ax = axes[ai]
        vals = mean_df[col].values if col in mean_df.columns else np.zeros(len(methods))
        stds = std_df[col].values  if col in std_df.columns  else np.zeros(len(methods))
        bars = ax.bar(x, vals, color=colors, alpha=0.85, yerr=stds, capsize=3)
        # Highlight baseline
        if BASELINE in methods:
            bi = methods.index(BASELINE)
            bars[bi].set_edgecolor("crimson"); bars[bi].set_linewidth(2.5)
        ax.set_xticks(x)
        ax.set_xticklabels([m[:35] for m in methods], rotation=45, ha="right", fontsize=7)
        ax.set_ylabel(col.upper())
        ax.set_title(f"Mean ± std {col.upper()} (sample level, 5-fold CV)")
        ax.set_ylim(0, 1.05)
    plt.tight_layout()
    fn = os.path.join(FIGURES_DIR, "summary_bar.png")
    plt.savefig(fn, dpi=120, bbox_inches="tight"); plt.show(); print(f"Saved → {fn}")

print("§10 complete ✅")


## § 11 · Training Curves

Per-fold fine-tuning history from `history_finetune.csv`. Useful to spot overfitting (train AUC ~ 1.0 while val AUC is low)
or under-training (AUC still rising at the last epoch — increase `EPOCHS_FT`).


In [ ]:
# § 11 — Training curves: loss and AUC per fold from history_finetune.csv
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
FIGURES_DIR = globals().get("FIGURES_DIR", os.path.join(LOCAL_RUNS, "figures"))
os.makedirs(FIGURES_DIR, exist_ok=True)

folds = sorted(d for d in os.listdir(LOCAL_RUNS)
               if d.startswith("fold") and os.path.exists(os.path.join(LOCAL_RUNS, d, "history_finetune.csv")))
if not folds:
    print("No history_finetune.csv files found — run §9 first."); raise SystemExit()

K = len(folds)
plot_cols = []
sample_hist = pd.read_csv(os.path.join(LOCAL_RUNS, folds[0], "history_finetune.csv"))
for c in sample_hist.columns:
    if any(kw in c for kw in ("loss","auc","acc")) and "val_" not in c:
        plot_cols.append(c)

nplots = min(len(plot_cols), 4)
fig, axes = plt.subplots(K, nplots, figsize=(4*nplots, 3*K), squeeze=False)
for fi, fold_dir in enumerate(folds):
    hist = pd.read_csv(os.path.join(LOCAL_RUNS, fold_dir, "history_finetune.csv"))
    for pi, col in enumerate(plot_cols[:nplots]):
        ax = axes[fi][pi]
        ax.plot(hist[col], label="train")
        vcol = "val_" + col
        if vcol in hist.columns:
            ax.plot(hist[vcol], ls="--", label="val")
        ax.set_title(f"{fold_dir} — {col}", fontsize=8)
        ax.set_xlabel("epoch"); ax.legend(fontsize=6)

plt.suptitle("Fine-tuning history per fold", fontsize=11)
plt.tight_layout()
fn = os.path.join(FIGURES_DIR, "training_curves.png")
plt.savefig(fn, dpi=100, bbox_inches="tight"); plt.show(); print(f"Saved → {fn}")
print("§11 complete ✅")


## § 12 · Per-Fold Results: Proposed vs Colour Baseline

Shows paired per-fold delta (proposed − baseline) for AUC and F1(fake). With only 5 folds and 11–13 samples each,
no significance test is meaningful — report the delta pattern and note if the proposed method consistently beats the baseline or not.

**Red flag**: proposed method consistently below the colour baseline → deep model is not learning beyond colour.


In [ ]:
# § 12 — Per-fold results table: proposed vs baseline, with deltas
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
FIGURES_DIR = globals().get("FIGURES_DIR", os.path.join(LOCAL_RUNS, "figures"))
os.makedirs(FIGURES_DIR, exist_ok=True)

PROPOSED = "PROPOSED: Wombat-OA + weighted beam + log-prob fusion"
BASELINE = "colour-statistics random forest (baseline)"
KEYS     = ["auc", "f1_fake", "acc", "recall_fake", "specificity"]

folds = sorted(d for d in os.listdir(LOCAL_RUNS)
               if d.startswith("fold") and os.path.exists(os.path.join(LOCAL_RUNS, d, "ablation_sample_level.csv")))
if not folds:
    print("No ablation_sample_level.csv found — run §9 first."); raise SystemExit()

rows_prop, rows_base = [], []
for fd in folds:
    df = pd.read_csv(os.path.join(LOCAL_RUNS, fd, "ablation_sample_level.csv"), index_col=0)
    for rows, name in [(rows_prop, PROPOSED), (rows_base, BASELINE)]:
        if name in df.index:
            rows.append({k: df.loc[name, k] for k in KEYS if k in df.columns})
        else:
            rows.append({k: float("nan") for k in KEYS})

prop_df = pd.DataFrame(rows_prop, index=folds)
base_df = pd.DataFrame(rows_base, index=folds)
delta_df = prop_df - base_df

combined = pd.concat([prop_df.add_suffix("_prop"), base_df.add_suffix("_base"), delta_df.add_suffix("_Δ")], axis=1)
print("Per-fold results (sample level): proposed | baseline | Δ(proposed − baseline)")
display(combined.round(4))

# Line plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
K = len(folds)
for ai, col in enumerate(["auc", "f1_fake"]):
    ax = axes[ai]
    ax.plot(range(K), prop_df[col].values, "o-", label="Proposed", color="steelblue")
    ax.plot(range(K), base_df[col].values, "s--", label="Colour baseline", color="crimson")
    ax.set_xticks(range(K)); ax.set_xticklabels(folds, rotation=15)
    ax.set_ylabel(col.upper()); ax.set_title(f"{col.upper()} per fold (sample level)")
    ax.set_ylim(0, 1.05); ax.legend()
plt.tight_layout()
fn = os.path.join(FIGURES_DIR, "perfold_comparison.png")
plt.savefig(fn, dpi=120, bbox_inches="tight"); plt.show(); print(f"Saved → {fn}")
print("§12 complete ✅")


## § 13 · Pooled Out-of-Fold Analysis

Every sample is tested exactly once (different fold). Uses each fold's tuned threshold (as `run_cv.py` does) before pooling.
Shows ROC, PR curves, calibration diagram, and sample-level confusion matrices for 4 key methods.

**Why pooled OOF matters**: fold-level tables have ~11–13 samples each (very noisy). The pooled OOF table uses all samples
and is the most reliable summary when folds are small.


In [ ]:
# § 13 — Pooled OOF: ROC, PR, calibration, confusion matrices (4 methods, sample level)
import os, sys, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix
from scipy.special import expit, logit
from IPython.display import display

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
FIGURES_DIR = globals().get("FIGURES_DIR", os.path.join(LOCAL_RUNS, "figures"))
sys.path.insert(0, os.path.join(COLAB_ROOT, "src"))
os.makedirs(FIGURES_DIR, exist_ok=True)
import wombat_beam as wb

PROPOSED = "PROPOSED: Wombat-OA + weighted beam + log-prob fusion"
BASELINE = "colour-statistics random forest (baseline)"
TTA      = "TTA mean-logit, hafm"
BEST_SGL = "single view, hafm"
METHODS  = [BASELINE, BEST_SGL, TTA, PROPOSED]

folds = sorted(d for d in os.listdir(LOCAL_RUNS)
               if d.startswith("fold") and os.path.exists(os.path.join(LOCAL_RUNS, d, "preds_test.csv")))
if not folds:
    print("preds_test.csv not found — run §9 first."); raise SystemExit()

parts = []
for fd in folds:
    q  = pd.read_csv(os.path.join(LOCAL_RUNS, fd, "preds_test.csv"))
    th = json.load(open(os.path.join(LOCAL_RUNS, fd, "thresholds.json")))
    for m in th:
        if m not in q.columns: continue
        pm = q[m].clip(1e-6, 1 - 1e-6)
        t  = min(max(th[m], 1e-6), 1 - 1e-6)
        q[m] = expit(logit(pm) - logit(t))   # shift so threshold = 0.5
    parts.append(q)
P = pd.concat(parts, ignore_index=True)

# Sample-level aggregation
def agg(col):
    ps, ys = wb.aggregate_by_group(P[col].values, P.y.values, P.group.values)
    return ps, ys

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for mi, m in enumerate(METHODS):
    if m not in P.columns:
        print(f"⚠ Method '{m}' not in pooled predictions — skip"); continue
    ps, ys = agg(m)
    label = m[:30]

    # ROC
    fpr, tpr, _ = roc_curve(ys, ps)
    roc_auc = auc(fpr, tpr)
    axes[0][mi].plot(fpr, tpr, lw=2, label=f"AUC={roc_auc:.3f}")
    axes[0][mi].plot([0,1],[0,1],"k--",lw=0.8)
    axes[0][mi].set_xlabel("FPR"); axes[0][mi].set_ylabel("TPR")
    axes[0][mi].set_title(f"ROC: {label}", fontsize=8)
    axes[0][mi].legend(fontsize=7)

    # PR
    prec, rec, _ = precision_recall_curve(ys, ps)
    pr_auc = auc(rec, prec)
    axes[1][mi].plot(rec, prec, lw=2, label=f"AP={pr_auc:.3f}")
    axes[1][mi].set_xlabel("Recall"); axes[1][mi].set_ylabel("Precision")
    axes[1][mi].set_title(f"PR: {label}", fontsize=8)
    axes[1][mi].legend(fontsize=7)

plt.tight_layout()
fn = os.path.join(FIGURES_DIR, "roc_pr_curves.png")
plt.savefig(fn, dpi=120, bbox_inches="tight"); plt.show(); print(f"Saved → {fn}")

# Calibration + confusion matrices
fig2, axes2 = plt.subplots(2, 4, figsize=(18, 9))
for mi, m in enumerate(METHODS):
    if m not in P.columns: continue
    ps, ys = agg(m)
    bins = np.linspace(0, 1, 11)
    bin_ids = np.digitize(ps, bins) - 1
    mean_conf, mean_acc = [], []
    for b in range(10):
        mask = bin_ids == b
        if mask.sum() > 0:
            mean_conf.append(ps[mask].mean()); mean_acc.append(ys[mask].mean())
    axes2[0][mi].plot([0,1],[0,1],"k--",lw=0.8,label="perfect")
    axes2[0][mi].scatter(mean_conf, mean_acc, s=40, c="steelblue")
    axes2[0][mi].set_xlabel("Mean confidence"); axes2[0][mi].set_ylabel("Fraction positive")
    axes2[0][mi].set_title(f"Calibration: {m[:28]}", fontsize=7)
    axes2[0][mi].legend(fontsize=6)

    cm = confusion_matrix(ys, (ps >= 0.5).astype(int))
    axes2[1][mi].imshow(cm, cmap="Blues", vmin=0)
    for r in range(2):
        for c in range(2):
            axes2[1][mi].text(c, r, cm[r,c], ha="center", va="center", fontsize=12, color="black")
    axes2[1][mi].set_xticks([0,1]); axes2[1][mi].set_yticks([0,1])
    axes2[1][mi].set_xticklabels(["Real","Fake"]); axes2[1][mi].set_yticklabels(["Real","Fake"])
    axes2[1][mi].set_xlabel("Predicted"); axes2[1][mi].set_ylabel("True")
    axes2[1][mi].set_title(f"Confusion: {m[:28]}", fontsize=7)

plt.tight_layout()
fn2 = os.path.join(FIGURES_DIR, "calibration_confusion.png")
plt.savefig(fn2, dpi=120, bbox_inches="tight"); plt.show(); print(f"Saved → {fn2}")
print("§13 complete ✅")


## § 14 · Tuned Decoder Parameters

Wombat-OA finds 13 parameters per fold (head weights `a_cnn/gnn/hafm`, temperatures `τ`, view reliability `β`,
diversity penalty `γ`, length-normalisation `α`, beam width `B`, depth `L`, sharpness `κ`, threshold `θ`).
High variance across folds means the optimiser is overfitting a small val set — the "untuned" rows in the ablation table provide a sanity check.


In [ ]:
# § 14 — Tuned Wombat-OA decoder parameters per fold
import os, json
import numpy as np, pandas as pd
from IPython.display import display

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
FIGURES_DIR = globals().get("FIGURES_DIR", os.path.join(LOCAL_RUNS, "figures"))
os.makedirs(FIGURES_DIR, exist_ok=True)

folds = sorted(d for d in os.listdir(LOCAL_RUNS)
               if d.startswith("fold") and os.path.exists(os.path.join(LOCAL_RUNS, d, "wombat_params.json")))
if not folds:
    print("wombat_params.json not found — run §9 first."); raise SystemExit()

rows = {}
for fd in folds:
    p = json.load(open(os.path.join(LOCAL_RUNS, fd, "wombat_params.json")))
    flat = {}
    for k, v in p.items():
        if hasattr(v, "__iter__") and not isinstance(v, str):
            for i, vi in enumerate(v):
                flat[f"{k}[{i}]"] = round(float(vi), 4)
        elif isinstance(v, (int, float)):
            flat[k] = round(float(v), 4)
    rows[fd] = flat

df = pd.DataFrame(rows).T
print("Tuned Wombat-OA parameters per fold:")
display(df.style.highlight_max(axis=0, color="lightgreen").highlight_min(axis=0, color="lightyellow")
        .format("{:.4f}"))

print(f"\nParameter std across folds (stability check):")
print(df.std(ddof=0).round(4).to_string())
print("\nHigh std → optimiser overfits the small val set. Use 'untuned' row as sanity check.")
print("§14 complete ✅")


## § 15 · Uncertainty Check

Tests whether MC-Dropout uncertainty (std of logits over T=16 samples) predicts misclassification.
- **AUROC ≈ 0.5**: uncertainty is uninformative — the MC-Dropout claim is unsupported on this data.
- **AUROC > 0.6**: uncertainty is somewhat predictive — useful for risk-coverage curves.

Also shows a risk-coverage curve at the sample level: what accuracy you get if you refer the most uncertain X% of samples to a human expert.


In [ ]:
# § 15 — Uncertainty check: AUROC of (MC std predicts misclassification) + risk-coverage
import os, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
FIGURES_DIR = globals().get("FIGURES_DIR", os.path.join(LOCAL_RUNS, "figures"))
sys.path.insert(0, os.path.join(COLAB_ROOT, "src"))
os.makedirs(FIGURES_DIR, exist_ok=True)
import wombat_beam as wb

folds = sorted(d for d in os.listdir(LOCAL_RUNS)
               if d.startswith("fold") and os.path.exists(os.path.join(LOCAL_RUNS, d, "cache_test.npz")))
if not folds:
    print("cache_test.npz not found — run §9 first."); raise SystemExit()

aucs_unc = []
ps_all, ys_all, unc_all = [], [], []
for fd in folds:
    c = wb.Cache.load(os.path.join(LOCAL_RUNS, fd, "cache_test.npz"))
    N, V, H = c.shape
    # mean uncertainty across views and heads (hafm only = head index 2)
    unc = c.U.mean(axis=(1,2))   # (N,) — mean MC std
    # default decode for probability
    from wombat_beam import default_params, unpack, decode
    p_obj = unpack(default_params(H, V), H, V)
    prob = decode(c, p_obj)
    misclf = (((prob >= 0.5).astype(int)) != c.y).astype(int)
    if len(np.unique(misclf)) > 1:
        auc_unc = roc_auc_score(misclf, unc)
    else:
        auc_unc = float("nan")
    aucs_unc.append(auc_unc)
    ps_all.append(prob); ys_all.append(c.y); unc_all.append(unc)

print(f"AUROC(uncertainty → misclassification) per fold: {[round(a,3) for a in aucs_unc]}")
print(f"Mean: {np.nanmean(aucs_unc):.3f}  |  ~0.5 means uncertainty is uninformative.")

# Risk-coverage curve (pooled)
ps_pool = np.concatenate(ps_all); ys_pool = np.concatenate(ys_all); unc_pool = np.concatenate(unc_all)
# Sample level
all_unc = unc_pool
order = np.argsort(all_unc)   # most certain first
accs, coverages = [], []
for frac in np.linspace(0.1, 1.0, 20):
    n_covered = max(1, int(frac * len(order)))
    idx = order[:n_covered]
    pred = (ps_pool[idx] >= 0.5).astype(int)
    acc = (pred == ys_pool[idx]).mean()
    accs.append(acc); coverages.append(frac)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(range(len(folds)), aucs_unc, color="steelblue")
axes[0].axhline(0.5, ls="--", c="grey", label="chance"); axes[0].legend()
axes[0].set_xticks(range(len(folds))); axes[0].set_xticklabels(folds)
axes[0].set_ylabel("AUROC"); axes[0].set_title("Uncertainty → misclassification AUROC per fold")

axes[1].plot(coverages, accs, "o-", color="steelblue")
axes[1].set_xlabel("Coverage (fraction of samples kept)"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Risk-coverage curve (most certain first)")
axes[1].set_ylim(0, 1.05); axes[1].invert_xaxis()

plt.tight_layout()
fn = os.path.join(FIGURES_DIR, "uncertainty.png")
plt.savefig(fn, dpi=120, bbox_inches="tight"); plt.show(); print(f"Saved → {fn}")
print("§15 complete ✅")


## § 16 · Per-Type Breakdown (fake samples only)

Fake saffron in this dataset is labelled by type (`-`, `C`, `S`, `M` from the filename).
The meaning of C/S/M is **not documented in the dataset files** — confirm before drawing conclusions.
This breakdown shows whether the model struggles more with one fake type than another.


In [ ]:
# § 16 — Per-type breakdown for fake samples (types -, C, S, M) at sample level
import os, sys, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.special import expit, logit
from IPython.display import display

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_DATA  = globals().get("LOCAL_DATA", os.path.join(COLAB_ROOT, "data"))
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
FIGURES_DIR = globals().get("FIGURES_DIR", os.path.join(LOCAL_RUNS, "figures"))
sys.path.insert(0, os.path.join(COLAB_ROOT, "src"))
os.makedirs(FIGURES_DIR, exist_ok=True)
import wombat_beam as wb

PROPOSED = "PROPOSED: Wombat-OA + weighted beam + log-prob fusion"
BASELINE = "colour-statistics random forest (baseline)"

man = pd.read_csv(os.path.join(LOCAL_DATA, "manifest.csv"))
# sample → type (fake only)
samp_type = man[man.label==1].groupby("group")["type"].first()

folds = sorted(d for d in os.listdir(LOCAL_RUNS)
               if d.startswith("fold") and os.path.exists(os.path.join(LOCAL_RUNS, d, "preds_test.csv")))
if not folds:
    print("preds_test.csv not found — run §9 first."); raise SystemExit()

parts = []
for fd in folds:
    q  = pd.read_csv(os.path.join(LOCAL_RUNS, fd, "preds_test.csv"))
    th = json.load(open(os.path.join(LOCAL_RUNS, fd, "thresholds.json")))
    for m in th:
        if m not in q.columns: continue
        pm = q[m].clip(1e-6, 1 - 1e-6); t = min(max(th[m],1e-6),1-1e-6)
        q[m] = expit(logit(pm) - logit(t))
    parts.append(q)
P = pd.concat(parts, ignore_index=True)

# Sample-level aggregation
results = {}
for m in [PROPOSED, BASELINE]:
    if m not in P.columns: continue
    ps, ys = wb.aggregate_by_group(P[m].values, P.y.values, P.group.values)
    groups, _ = np.unique(P.group.values, return_inverse=True)
    # unique groups from agg
    u_groups = np.unique(P.group.values)
    # map group → type for fake samples
    type_map = samp_type.to_dict()
    results[m] = []
    for g, p_g, y_g in zip(u_groups, ps, ys):
        t = type_map.get(g, "real")
        results[m].append({"group":g,"p":p_g,"y":y_g,"type":t})

for m, rows in results.items():
    df = pd.DataFrame(rows)
    fake_df = df[df.y==1].copy()
    fake_df["correct"] = (fake_df.p >= 0.5).astype(int)
    print(f"\n=== {m[:50]} — fake sample recall by type ===")
    print(fake_df.groupby("type")["correct"].agg(["sum","count","mean"]).rename(
        columns={"sum":"correct","count":"total","mean":"recall"}
    ).to_string())
    print("⚠ Type meanings (C/S/M) are unconfirmed — interpret with caution.")

print("§16 complete ✅")


## § 17 · Optimiser Study: Wombat-OA vs Random Search

Runs both optimisers with 10 independent seeds on each fold's validation cache, then pools results.
Shows convergence curves and a final box plot, and reports the one-sided Wilcoxon signed-rank p-value
(H1: Wombat-OA reaches lower fitness than random search with the same budget).

**Note**: with small val caches (~11–13 samples), convergence curves may be noisy. The Wilcoxon test requires 5+ seeds.


In [ ]:
# § 17 — Optimiser study: Wombat-OA vs Random Search, 10 seeds × each fold's val cache
import os, sys, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
FIGURES_DIR = globals().get("FIGURES_DIR", os.path.join(LOCAL_RUNS, "figures"))
POP  = globals().get("POP", 20)
ITERS= globals().get("ITERS", 25)
sys.path.insert(0, os.path.join(COLAB_ROOT, "src"))
os.makedirs(FIGURES_DIR, exist_ok=True)
import wombat_beam as wb

N_OPT_SEEDS = 10

folds = sorted(d for d in os.listdir(LOCAL_RUNS)
               if d.startswith("fold") and os.path.exists(os.path.join(LOCAL_RUNS, d, "cache_val.npz")))
if not folds:
    print("cache_val.npz not found — run §9 first."); raise SystemExit()

# Cache result to Drive to avoid re-running
cmp_cache = os.path.join(LOCAL_RUNS, "optimizer_comparison.json")
if os.path.exists(cmp_cache):
    print(f"Loading cached optimiser comparison from {cmp_cache}")
    cmp = json.load(open(cmp_cache))
    for k in ("wombat","random"):
        cmp[k] = np.array(cmp[k])
    cmp["hist_wombat"] = [np.array(h) for h in cmp["hist_wombat"]]
    cmp["hist_random"] = [np.array(h) for h in cmp["hist_random"]]
else:
    print(f"Running optimiser comparison: {N_OPT_SEEDS} seeds × {len(folds)} folds ...")
    all_fw, all_fr, all_hw, all_hr = [], [], [], []
    for fd in folds:
        val = wb.Cache.load(os.path.join(LOCAL_RUNS, fd, "cache_val.npz"))
        print(f"  {fd}: val shape {val.shape}", flush=True)
        res = wb.compare_optimizers(val, seeds=range(N_OPT_SEEDS), pop=POP, iters=ITERS)
        all_fw.extend(res["wombat"].tolist()); all_fr.extend(res["random"].tolist())
        all_hw.extend(res["hist_wombat"]); all_hr.extend(res["hist_random"])
    cmp = {"wombat": np.array(all_fw), "random": np.array(all_fr),
           "hist_wombat": all_hw, "hist_random": all_hr}
    cmp["wilcoxon_p"] = float(wilcoxon(cmp["wombat"], cmp["random"], alternative="less").pvalue)         if len(cmp["wombat"]) >= 5 and not np.allclose(cmp["wombat"], cmp["random"]) else float("nan")
    # Save
    save = {"wombat": cmp["wombat"].tolist(), "random": cmp["random"].tolist(),
            "hist_wombat": [h.tolist() for h in cmp["hist_wombat"]],
            "hist_random": [h.tolist() for h in cmp["hist_random"]],
            "wilcoxon_p": cmp["wilcoxon_p"]}
    json.dump(save, open(cmp_cache, "w"))
    print(f"Saved → {cmp_cache}")

print(f"\nWombat-OA: {np.mean(cmp['wombat']):.4f} ± {np.std(cmp['wombat']):.4f}")
print(f"Random:    {np.mean(cmp['random']):.4f} ± {np.std(cmp['random']):.4f}")
print(f"Wilcoxon p (H1: Wombat < Random): {cmp.get('wilcoxon_p', float('nan')):.4g}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for h in cmp["hist_wombat"]: axes[0].plot(np.linspace(0,1,len(h)), h, c="tab:green", alpha=0.3)
for h in cmp["hist_random"]:  axes[0].plot(np.linspace(0,1,len(h)), h, c="tab:gray",  alpha=0.3)
axes[0].set_title("Convergence (green=Wombat-OA, gray=Random)")
axes[0].set_xlabel("Fraction of budget"); axes[0].set_ylabel("Best val fitness")
axes[1].boxplot([cmp["wombat"], cmp["random"]], tick_labels=["Wombat-OA","Random"])
axes[1].set_title(f"Final fitness ({N_OPT_SEEDS}×{len(folds)} seeds)")
plt.tight_layout()
fn = os.path.join(FIGURES_DIR, "optimizer_study.png")
plt.savefig(fn, dpi=120, bbox_inches="tight"); plt.show(); print(f"Saved → {fn}")
print("§17 complete ✅")


## § 18 · Headline Dashboard

In [ ]:
# § 18 — Headline dashboard: best method, vs baseline, honest auto-generated summary
import os, sys
import numpy as np, pandas as pd
from IPython.display import display

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
sys.path.insert(0, os.path.join(COLAB_ROOT, "src"))
import wombat_beam as wb

PROPOSED = "PROPOSED: Wombat-OA + weighted beam + log-prob fusion"
BASELINE = "colour-statistics random forest (baseline)"

mean_fp = os.path.join(LOCAL_RUNS, "summary_sample_mean.csv")
if not os.path.exists(mean_fp):
    print("summary_sample_mean.csv not found — run §10 first."); raise SystemExit()
mean_df = pd.read_csv(mean_fp, index_col=0)

folds = sorted(d for d in os.listdir(LOCAL_RUNS)
               if d.startswith("fold") and os.path.exists(os.path.join(LOCAL_RUNS, d, "ablation_sample_level.csv")))
if not folds:
    print("No fold ablation files — run §9 first."); raise SystemExit()
tabs = [pd.read_csv(os.path.join(LOCAL_RUNS, d, "ablation_sample_level.csv"), index_col=0) for d in folds]
cat = pd.concat(tabs)
std_df = cat.groupby(level=0, sort=False)[["auc","f1_fake","acc"]].std(ddof=0)

# Best method by mean AUC
best_row = mean_df["auc"].idxmax() if "auc" in mean_df.columns else None
prop_auc  = mean_df.loc[PROPOSED, "auc"]  if PROPOSED  in mean_df.index else float("nan")
base_auc  = mean_df.loc[BASELINE, "auc"]  if BASELINE  in mean_df.index else float("nan")
prop_f1   = mean_df.loc[PROPOSED, "f1_fake"] if PROPOSED in mean_df.index else float("nan")
base_f1   = mean_df.loc[BASELINE, "f1_fake"] if BASELINE in mean_df.index else float("nan")
prop_std  = std_df.loc[PROPOSED, "auc"]   if PROPOSED in std_df.index else float("nan")

print("=" * 60)
print("HEADLINE RESULTS (sample-level, 5-fold group CV)")
print("=" * 60)
print(f"Best method by mean AUC: {best_row}")
print(f"  Proposed mean AUC:     {prop_auc:.3f}  (std {prop_std:.3f})")
print(f"  Colour baseline AUC:   {base_auc:.3f}")
print(f"  Proposed mean F1(fake):{prop_f1:.3f}")
print(f"  Colour baseline F1:    {base_f1:.3f}")
print()

# Auto-generated honest summary
beats_auc = prop_auc > base_auc if not (np.isnan(prop_auc) or np.isnan(base_auc)) else None
beats_f1  = prop_f1 > base_f1  if not (np.isnan(prop_f1)  or np.isnan(base_f1))  else None
delta_auc = prop_auc - base_auc if beats_auc is not None else float("nan")
delta_f1  = prop_f1  - base_f1  if beats_f1  is not None else float("nan")

lines = []
if beats_auc is True:
    lines.append(f"✅ PROPOSED beats the colour baseline on mean sample-level AUC by {delta_auc:+.3f}.")
elif beats_auc is False:
    lines.append(f"⚠  PROPOSED does NOT beat the colour baseline on mean sample-level AUC ({delta_auc:+.3f}).")
else:
    lines.append("⚠  Could not compare proposed vs baseline (missing data).")

if beats_f1 is True:
    lines.append(f"✅ PROPOSED beats the colour baseline on mean F1(fake) by {delta_f1:+.3f}.")
elif beats_f1 is False:
    lines.append(f"⚠  PROPOSED does NOT beat the colour baseline on F1(fake) ({delta_f1:+.3f}).")

if not np.isnan(prop_std):
    if prop_std > 0.15:
        lines.append(f"⚠  High fold-to-fold std on AUC ({prop_std:.3f}) — results are noisy with only {len(folds)} folds of ~11–13 samples.")
    else:
        lines.append(f"ℹ  Fold std on AUC: {prop_std:.3f} (across {len(folds)} folds).")
lines.append("ℹ  59 independent samples is a small dataset — use mean ± std in the paper, not single folds.")

for line in lines:
    print(line)
print("=" * 60)
print("§18 complete ✅")


## § 19 · Package Results

In [ ]:
# § 19 — Zip csv/json/png results from runs/cv1 to Drive; offer files.download
import os, shutil, glob, time

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
DRIVE_RUNS_DIR = globals().get("DRIVE_RUNS_DIR", "")

ts = time.strftime("%Y%m%d_%H%M")
zip_name = f"saffron_results_{ts}"
zip_path_base = os.path.join(COLAB_ROOT, zip_name)

# Collect only lightweight files (no weights)
tmp = os.path.join(COLAB_ROOT, "results_tmp"); os.makedirs(tmp, exist_ok=True)
for dp, dns, fns in os.walk(LOCAL_RUNS):
    for fn in fns:
        ext = os.path.splitext(fn)[1].lower()
        if ext not in (".csv", ".json", ".png"): continue
        src = os.path.join(dp, fn)
        rel = os.path.relpath(src, LOCAL_RUNS)
        dst = os.path.join(tmp, rel); os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)

zip_path = shutil.make_archive(zip_path_base, "zip", tmp)
print(f"Created results zip: {zip_path}")

# Copy to Drive
if DRIVE_RUNS_DIR:
    os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)
    drive_zip = os.path.join(DRIVE_RUNS_DIR, os.path.basename(zip_path))
    shutil.copy2(zip_path, drive_zip)
    try: os.sync()
    except AttributeError: pass
    print(f"Copied to Drive: {drive_zip}")

# Offer download in Colab
try:
    from google.colab import files as _cfiles
    _cfiles.download(zip_path)
    print("Download triggered ✅")
except (ModuleNotFoundError, Exception) as e:
    print(f"Colab download not available ({e}). Find the zip at: {zip_path}")

shutil.rmtree(tmp, ignore_errors=True)
print("§19 complete ✅")


## § 20 · Interpretation Guide

### How to read the tables
- **Primary table**: sample-level, group-CV (§10 summary, §18 dashboard). One decision per physical sample (all shots fused).
- **Secondary**: image-level table (less meaningful because the same sample appears multiple times).
- **Pooled OOF** (§13): every sample is tested exactly once — the most reliable aggregate with only 11–13 samples per fold.
- Compare **every deep-learning row** against the `colour-statistics random forest (baseline)` row.
  A deep model below this baseline is not learning anything beyond colour.

### Red flags
| Symptom | Likely cause |
|---------|-------------|
| Deep model far below colour baseline | ImageNet weights not loaded; too few epochs; overfitting |
| Train AUC ~1.0, val AUC low | Overfitting; try more dropout, fewer epochs, larger L2 |
| Huge fold variance (std > 0.15) | Expected with 11–13 samples/fold; report mean ± std, not single folds |
| 100% accuracy on any fold | Dataset leakage or label bug; re-check manifest |
| Uncertainty AUROC ≈ 0.5 | MC-Dropout claim unsupported; don't claim calibrated uncertainty |

### Honest limits (from README § 6)
- 59 independent samples is small. Tuning 13 decoder parameters on ~12 val samples is noisy.
- The meaning of C / S / M fake types is not documented — confirm before type-level analysis.
- For two classes, log-probability fusion equals weighted logit averaging. Gains must come from tuned weights, diversity-aware beam, or the optimiser — the ablation rows isolate each.
- Do **not** report results on the original Training/Testing split (leaked, §5 reports this).


## § 21 · Troubleshooting

| Problem | Solution |
|---------|---------|
| **No GPU** | Runtime → Change runtime type → T4 GPU. If unavailable, set `ALLOW_CPU=True` in §2 (very slow). |
| **OOM during training** | Set `BATCH=8` and/or `IMG_SIZE=224` in §2. |
| **Drive zips not found** | Check `DRIVE_DATA_DIR` in §2 points to the folder with the two `.zip` files. |
| **Keras/TF version mismatch** | §1 installs the needed versions automatically. **Restart the runtime once** after the install message, then re-run from §1. |
| **Session disconnect** | Re-run §1–§3 (environment, config, drive), then §8 (resume: rsyncs Drive → local). §9 skips finished folds. |
| **`prepare_dataset.py` fails** | Check that the zip files contain `Training/{Fake,Real}` and `Testing/{Fake,Real}` subdirectories. |
| **"cache/manifest order mismatch"** | Delete `runs/cv1/fold*` and re-run §9. Manifest and caches must come from the same `prepare_dataset.py` run. |
| **Smoke test slow** | Expected — smoke uses CPU by default. On GPU it takes ~5–15 min. Numbers are not results. |


## § 22 · Parameters Reference

| Variable | Default | Effect of changing |
|----------|---------|-------------------|
| `DRIVE_DATA_DIR` | `"/content/drive/MyDrive/saffron_data"` | Folder in Drive with the two dataset zips |
| `DRIVE_RUNS_DIR` | `"/content/drive/MyDrive/saffron_runs"` | Drive folder for checkpoint backup |
| `N_FOLDS` | `5` | Number of CV folds. Changing this invalidates cached caches. |
| `IMG_SIZE` | `256` | Input resolution. `384` gives better results but needs `BATCH=8`. |
| `BATCH` | `16` | Mini-batch size. Use `8` if GPU < 16 GB. |
| `EPOCHS_WARM` | `3` | Warm-up epochs (backbone frozen). Rarely needs changing. |
| `EPOCHS_FT` | `25` | Fine-tune epochs. Increase to `40` if val AUC is still rising. |
| `MC_SAMPLES` | `16` | MC-Dropout samples at cache time. `32` for better uncertainty estimates. |
| `POP` | `20` | Wombat-OA population size. |
| `ITERS` | `25` | Wombat-OA iterations. |
| `SEED` | `42` | Global random seed. |
| `SMOKE` | `False` | Pipeline test mode: fold 0 only, tiny settings. Numbers meaningless. |
| `ALLOW_CPU` | `False` | Allow running without GPU. Very slow. |
| `SAVE_WEIGHTS_TO_DRIVE` | `False` | Include `*.weights.h5` in Drive sync (~300 MB/fold). |


## § 24 · Leave-One-Camera-Out (LOCO) Evaluation

Trains on ONE phone, validates on held-out samples of the same phone, and tests on the OTHER phone.
This is the strongest cross-device test available when both phones contain real and fake samples.

Roles encoded in the `fold` column so the standard runner is reused:
- `fold 0` = TEST (other camera)
- `fold 1` = VALIDATION (held-out groups of training camera)
- `fold 2` = TRAIN (rest of training camera)

> ⚠ Validation folds here are tiny (6–8 samples). Wombat-OA tuning is therefore noisy — always
> report results alongside the untuned-beam row.


In [ ]:
# § 24 — Leave-one-camera-out evaluation (both cross-device directions)
import os, sys, subprocess, shutil
import pandas as pd
from IPython.display import display

COLAB_ROOT   = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_DATA   = globals().get("LOCAL_DATA", os.path.join(COLAB_ROOT, "data"))
LOCAL_RUNS   = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
DRIVE_RUNS_DIR = globals().get("DRIVE_RUNS_DIR", "")
IMG_SIZE     = globals().get("IMG_SIZE", 256)
POP          = globals().get("POP", 20)
ITERS        = globals().get("ITERS", 25)
SEED         = globals().get("SEED", 42)

def run(cmd, cwd=COLAB_ROOT, env_extra=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2"}
    if env_extra: env.update(env_extra)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

# ── Step 1: generate LOCO manifests ─────────────────────────────────────────
manifest_path = os.path.join(LOCAL_DATA, "manifest.csv")
if not os.path.exists(manifest_path):
    print("manifest.csv not found — run §5 first."); raise SystemExit()

print("Generating LOCO manifests...")
run([sys.executable, "scripts/make_loco_manifests.py",
     "--manifest", manifest_path])

# Find generated manifests
import glob
loco_manifests = sorted(glob.glob(os.path.join(LOCAL_DATA, "manifest_loco_*_to_*.csv")))
print(f"Found {len(loco_manifests)} LOCO manifests: {[os.path.basename(m) for m in loco_manifests]}")

if not loco_manifests:
    print("No LOCO manifests generated — check that your data has >= 2 usable cameras."); raise SystemExit()

# ── Step 2: run cross-device test for each direction ─────────────────────────
loco_results = {}
for mf in loco_manifests:
    name = os.path.splitext(os.path.basename(mf))[0]   # manifest_loco_CAMA_to_CAMB
    parts = name.replace("manifest_loco_", "").split("_to_")
    out_name = f"loco_{parts[0]}_to_{parts[1]}"
    out_dir  = os.path.join(COLAB_ROOT, "runs", out_name)
    print(f"\n{'='*60}")
    print(f"Running LOCO: {out_name}")
    print(f"{'='*60}")
    run([sys.executable, "scripts/run_cv.py",
         "--manifest", mf,
         "--out", out_dir,
         "--folds", "0",
         "--img-size", str(IMG_SIZE),
         "--pop", str(POP),
         "--iters", str(ITERS),
         "--seed", str(SEED)])
    summary_path = os.path.join(out_dir, "summary_sample.csv")
    if not os.path.exists(summary_path):
        raise FileNotFoundError(f"LOCO summary was not generated: {summary_path}")
    if DRIVE_RUNS_DIR:
        drive_out = os.path.join(DRIVE_RUNS_DIR, out_name)
        os.makedirs(drive_out, exist_ok=True)
        shutil.copy2(summary_path, os.path.join(drive_out, "summary_sample.csv"))
        print(f"Synced LOCO summary to Drive: {drive_out}")
    loco_results[out_name] = out_dir

# ── Step 3: compare LOCO vs CV results ───────────────────────────────────────
print("\n" + "="*60)
print("LOCO vs Standard CV — PROPOSED method, sample level")
print("="*60)
PROPOSED = "PROPOSED: Wombat-OA + weighted beam + log-prob fusion"
comparison = []
cv_summary = os.path.join(LOCAL_RUNS, "summary_sample.csv")
if not os.path.exists(cv_summary):
    raise FileNotFoundError(f"Standard CV summary was not found: {cv_summary}")
cv = pd.read_csv(cv_summary, index_col=0)
if PROPOSED not in cv.index:
    raise KeyError(f"PROPOSED method is missing from {cv_summary}")
cv_row = cv.loc[PROPOSED]
comparison.append({
    "Evaluation": "Standard CV",
    "AUC": cv_row.get("auc", "—"),
    "F1 (fake)": cv_row.get("f1_fake", "—"),
    "Accuracy": cv_row.get("acc", "—"),
})
for out_name, out_dir in loco_results.items():
    summary_path = os.path.join(out_dir, "summary_sample.csv")
    df = pd.read_csv(summary_path, index_col=0)
    if PROPOSED not in df.index:
        raise KeyError(f"PROPOSED method is missing from {summary_path}")
    row = df.loc[PROPOSED]
    comparison.append({
        "Evaluation": out_name,
        "AUC": row.get("auc", "—"),
        "F1 (fake)": row.get("f1_fake", "—"),
        "Accuracy": row.get("acc", "—"),
    })

display(pd.DataFrame(comparison).set_index("Evaluation"))

print("\n§24 complete ✅")


## § 25 · Stress Test — Robustness to Image Degradation

Re-evaluates the SAVED fold models on 8 types of degraded test images **without retraining or retuning**.
This reveals:
- What features the network relies on (colour? texture? resolution?)
- Whether Wombat-OA beam decoding and log-probability fusion are more robust than a plain average

Perturbations applied to TEST images only:
| Name | Transform |
|------|-----------|
| `clean` | No change |
| `blur` | Gaussian blur σ=2 |
| `jpeg` | JPEG quality 20 |
| `noise` | Additive Gaussian noise std=12 |
| `lowres` | 4× downsample then upsample |
| `gray` | Grayscale (colour removed) |
| `hue` | Hue rotation 0.08 |
| `dark` | 40% darkening |

> ⚠ Needs `final.weights.h5` on disk (i.e. Colab session still alive, or `SAVE_WEIGHTS_TO_DRIVE=True`).


In [ ]:
# § 25 — Stress test: robustness of all methods to image degradation
import os, sys, subprocess
import pandas as pd
import matplotlib.pyplot as plt

COLAB_ROOT  = globals().get("COLAB_ROOT", "/content/saffron-authenticity")
LOCAL_DATA  = globals().get("LOCAL_DATA", os.path.join(COLAB_ROOT, "data"))
LOCAL_RUNS  = globals().get("LOCAL_RUNS", os.path.join(COLAB_ROOT, "runs", "cv1"))
FIGURES_DIR = globals().get("FIGURES_DIR", os.path.join(LOCAL_RUNS, "figures"))
os.makedirs(FIGURES_DIR, exist_ok=True)

manifest_path = os.path.join(LOCAL_DATA, "manifest.csv")
if not os.path.exists(manifest_path):
    print("manifest.csv not found — run §5 first."); raise SystemExit()

# Check at least one fold has weights
have_weights = any(
    os.path.exists(os.path.join(LOCAL_RUNS, f"fold{k}", "final.weights.h5")) or
    os.path.exists(os.path.join(LOCAL_RUNS, f"fold{k}", "best.weights.h5"))
    for k in range(5)
)
if not have_weights:
    print("⚠ No saved weights found in runs/cv1/fold*/. Weights are only kept if:")
    print("  (a) the Colab session is still alive (in-memory), or")
    print("  (b) you set SAVE_WEIGHTS_TO_DRIVE=True in §2 before training.")
    print("Skipping stress test.")
    raise SystemExit()

def run(cmd, cwd=COLAB_ROOT, env_extra=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2"}
    if env_extra: env.update(env_extra)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

print("Running stress test (all 8 perturbations) ...")
run([sys.executable, "scripts/stress_test.py",
     "--manifest", manifest_path,
     "--run", LOCAL_RUNS])

# ── Display results ──────────────────────────────────────────────────────────
results_csv = os.path.join(LOCAL_RUNS, "stress_results.csv")
if not os.path.exists(results_csv):
    print("stress_results.csv not found."); raise SystemExit()

from IPython.display import display
R = pd.read_csv(results_csv)
PERTURBS = ["clean","blur","jpeg","noise","lowres","gray","hue","dark"]

for level in ("sample", "image"):
    for metric in ("auc", "acc"):
        T = R[R.level == level].pivot(index="method", columns="perturbation", values=metric)
        T = T[[c for c in PERTURBS if c in T.columns]]
        print(f"\n=== {metric.upper()} — {level.upper()} level ===")
        display(T.round(3).style
                .background_gradient(cmap="RdYlGn", vmin=0, vmax=1)
                .format("{:.3f}"))

# ── Heatmap figure ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(R.method.unique()) * 0.45)))
for ai, metric in enumerate(("auc", "acc")):
    T = R[R.level == "sample"].pivot(index="method", columns="perturbation", values=metric)
    T = T[[c for c in PERTURBS if c in T.columns]]
    im = axes[ai].imshow(T.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    axes[ai].set_xticks(range(len(T.columns)))
    axes[ai].set_xticklabels(T.columns, rotation=45, ha="right", fontsize=8)
    axes[ai].set_yticks(range(len(T.index)))
    axes[ai].set_yticklabels([m[:40] for m in T.index], fontsize=7)
    for r in range(T.shape[0]):
        for c in range(T.shape[1]):
            axes[ai].text(c, r, f"{T.values[r,c]:.2f}",
                          ha="center", va="center", fontsize=6,
                          color="black" if T.values[r,c] > 0.5 else "white")
    axes[ai].set_title(f"{metric.upper()} by perturbation (sample level)")
    plt.colorbar(im, ax=axes[ai], shrink=0.8)

plt.tight_layout()
fn = os.path.join(FIGURES_DIR, "stress_test_heatmap.png")
plt.savefig(fn, dpi=120, bbox_inches="tight"); plt.show()
print(f"Saved → {fn}")
print("§25 complete ✅")


## § 26 · What to Do Next

1. **Copy numbers**: take `runs/cv1/summary_sample.csv` mean ± std rows into your paper table.
   Use the sample-level table (one decision per physical sample), not the image-level table.
2. **Run ≥ 3 training seeds**: change `SEED` in §2 and re-run §5–§9 with a fresh `DRIVE_RUNS_DIR`.
   Report mean ± std over seeds × folds.
3. **Compare against colour baseline**: if the proposed method does not beat the colour baseline
   on AUC and F1(fake), investigate (more epochs, larger `IMG_SIZE`, check ImageNet weights loaded).
4. **Confirm fake-type meanings**: identify what C, S, M mean in your dataset before publishing §16 results.
5. **Ablation table interpretation**: the beam, diversity penalty, and Wombat-OA rows exist to isolate
   each contribution — cite the ablation rows, not just the final proposed row.
6. **LOCO results (§24)**: report cross-device AUC for both directions. The G973F→J600G direction
   is harder (unseen adulterant types in the test camera).
7. **Stress test (§25)**: if accuracy drops to ~50% under grayscale, the model relies on colour —
   discuss this as a limitation (or strength, if colour is a valid discriminator for your application).
